# CortexHR attrition: the complete data-science workflow for a classification problem
Every numbered item of the supplied workflow (1-37) is a section below, with a **coverage table** (requirement -> where/how) followed by real code and outputs on the actual CortexHR attrition data.
Target: `left_within_12m` (1 = left within 12 months). Data: `03_Data/CortexHR_Dataset.xlsx` (fictional). Official helpers: `05_Starter/cortexhr_helpers.py`.
**Rules kept throughout:** the test set is untouched until Section 37; all learned preprocessing is fitted on training rows only; the validation vault is never used as input.

In [1]:
import sys, pathlib, inspect, time, warnings, subprocess
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / '05_Starter'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy.stats as st
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 100)
import cortexhr_helpers as H; H.DATA = ROOT / '03_Data' / 'csv'          # the OFFICIAL starter helper, unmodified
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold, GroupKFold, cross_val_predict, cross_validate, GridSearchCV,
                                     RandomizedSearchCV, train_test_split, learning_curve, validation_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, fbeta_score, accuracy_score,
                             balanced_accuracy_score, log_loss, brier_score_loss, matthews_corrcoef, cohen_kappa_score, confusion_matrix,
                             hamming_loss, make_scorer)
from cortexhr import config as C, data_loading as D, audit as A, features as F, modeling as M, evaluation as E, reporting as R, runs, workflow as W
SEED = C.SEED
print('Python', sys.version.split()[0], '| root', ROOT, '| official helper:', H.__file__.split('/')[-2:])

Python 3.14.8 | root C:\Users\USER\Downloads\cortexhr_phase1 | official helper: ['C:\\Users\\USER\\Downloads\\cortexhr_phase1\\05_Starter\\cortexhr_helpers.py']


---
---
# PHASE 1: BUSINESS AND PROBLEM UNDERSTANDING

---
## 1. Business Understanding

| Requirement | How it is covered in this notebook |
|---|---|
| Understand the business problem and requirements; identify the problem to solve | Cell 1.1: HR loses people it would like to keep; wants an early, ranked list of people at risk of leaving in 12 months |
| Why a machine-learning solution | Cell 1.2 compares a simple rule with ML on the same data |
| Stakeholders, end users, business objectives | Cell 1.1 table |
| Current process and its limitations | Cell 1.1 table |
| Expected business outcome; measurable success criteria | Cell 1.1 table + Cell 1.3 (thresholds checked at the end, Section 37) |
| Cost of incorrect predictions | Cell 1.3: explicit, adjustable cost assumptions and expected-cost table |
| Operational, legal, ethical, privacy constraints | Cell 1.1 table (human review only, protected attributes kept out of the model, DPDP-style privacy) |

**1.1 Business brief**

| Item | Answer |
|---|---|
| Problem | Valued employees leave unexpectedly; replacing them is slow and costly |
| Why ML | Many weak signals (pay position, raise age, engagement, workload) combine better in a model than in a manager's memory; a ranked list scales to thousands of people |
| Stakeholders / users | HR business partners and line managers (users); CHRO (sponsor); employees (affected); data/legal/privacy teams |
| Current process | Exit interviews after the fact; ad-hoc manager judgement; no ranked early-warning list |
| Expected outcome | A monthly ranked list; HR reviews the top group and acts on fixable causes (pay gap, stalled raise, workload) |
| Success criteria | Top-10% group captures at least 2x the leavers random selection would (lift >= 2); ROC-AUC >= 0.70; nothing leaks from the future; no automatic employment decisions |
| Constraints | Scores support a human decision only; protected attributes (age, gender, ...) are not model inputs and are used only for fairness checks; employee data stays internal; scores are not shown to the employee |

In [2]:
raw_csv = H.load('attrition_dataset')                                   # official loader (CSV export of the workbook)
raw = D.load_attrition()                                                # same table with typed dates (xlsx)
assert raw_csv.shape == raw.shape == (2389, 42); print('rows x columns:', raw.shape)
y_all = raw[C.TARGET]
print(f'attrition rate {100 * y_all.mean():.2f}%  ->  a 14.6% base rate means a random list of 10 people holds ~1.5 leavers')

rows x columns: (2389, 42)
attrition rate 14.61%  ->  a 14.6% base rate means a random list of 10 people holds ~1.5 leavers


In [3]:
# 1.2 Is ML better than a simple rule? Compare a hand-written 'stale raise' rule score with the model (cross-validated) later in 3.3.
rule_score = raw.months_since_last_raise.fillna(raw.months_since_last_raise.median())
print('single hand rule (months since last raise) ROC-AUC:', round(roc_auc_score(y_all, rule_score), 3))

single hand rule (months since last raise) ROC-AUC: 0.6


In [4]:
# 1.3 Cost of wrong predictions: explicit ASSUMPTIONS (change them to your company's numbers)
cost_fn, cost_fp = 5.0, 0.5       # cost units: missing a leaver (replacement, lost knowledge) vs a pointless review conversation
n, pos = len(raw), int(y_all.sum())
def expected_cost(flagged, caught): return (pos - caught) * cost_fn + (flagged - caught) * cost_fp
pd.DataFrame({'strategy': ['review nobody', 'review everybody', 'review a random 10%'],
              'flagged': [0, n, int(.1 * n)], 'leavers caught': [0, pos, int(.1 * pos)]}).assign(
    expected_cost=lambda d: [expected_cost(f, c) for f, c in zip(d.flagged, d['leavers caught'])])

,strategy,flagged,leavers caught,expected_cost
0,review nobody,0,0,1745.0
1,review everybody,2389,349,1020.0
2,review a random 10%,238,34,1677.0


---
## 2. Problem Definition

| Requirement | How it is covered in this notebook |
|---|---|
| Convert the business problem into an ML problem; confirm classification | Cell 2.1: predict a yes/no outcome per employee |
| Unit of prediction; prediction time; horizon | Cell 2.1: one employee at one `observation_date`; predict at that date; horizon 12 months |
| Target variable and meaning; positive and negative classes | Cell 2.1: `left_within_12m`, 1 = left, 0 = stayed |
| Binary / multiclass / multilabel / ordinal | Cell 2.2 verifies exactly two classes -> binary classification |
| Baseline performance and expected improvement | Cell 2.3: majority baseline and supplied benchmark sheet |
| Primary evaluation metric from business requirements | Cell 2.3: PR-AUC and top-10% recall/lift (business list), ROC-AUC as ranking check |

In [5]:
print('unit of prediction : one employee at one observation_date')
print('prediction time      :', raw.observation_date.min().date(), 'to', raw.observation_date.max().date(), '(snapshot date of each row)')
print('prediction horizon   : 12 months after observation_date')
print('target               :', C.TARGET, '| classes:', sorted(y_all.unique()), '| positive class = left (1)')

unit of prediction : one employee at one observation_date
prediction time      : 2021-01-01 to 2025-09-30 (snapshot date of each row)
prediction horizon   : 12 months after observation_date
target               : left_within_12m | classes: [np.int64(0), np.int64(1)] | positive class = left (1)


In [6]:
print('number of classes:', y_all.nunique(), '-> BINARY classification (not multiclass, multilabel or ordinal)')
print('label mapping kept:', {0: 'stayed', 1: 'left'})

number of classes: 2 -> BINARY classification (not multiclass, multilabel or ordinal)
label mapping kept: {0: 'stayed', 1: 'left'}


In [7]:
bench = H.load('baseline_benchmarks'); print(bench[['model', 'roc_auc_mean', 'pr_auc_mean', 'recall_mean', 'precision_mean', 'accuracy_mean']].round(3).to_string(index=False))
print(f'\nmajority baseline accuracy {100 * (1 - y_all.mean()):.1f}% with recall 0')
print('PRIMARY metrics: PR-AUC and top-10% recall/lift (the list HR can act on); SECONDARY: ROC-AUC, recall, precision. Accuracy is NOT used to choose models.')

                                            model  roc_auc_mean  pr_auc_mean  recall_mean  precision_mean  accuracy_mean
                             Dummy (always stays)         0.500        0.146        0.000           0.000          0.854
      Logistic Regression (class_weight=balanced)         0.713        0.332        0.630           0.260          0.683
                Decision Tree (depth 4, balanced)         0.650        0.256        0.524           0.234          0.676
              Random Forest (300 trees, balanced)         0.715        0.336        0.089           0.523          0.854
LEAKY Logistic Regression (adds exit information)         1.000        1.000        1.000           1.000          1.000

majority baseline accuracy 85.4% with recall 0
PRIMARY metrics: PR-AUC and top-10% recall/lift (the list HR can act on); SECONDARY: ROC-AUC, recall, precision. Accuracy is NOT used to choose models.


---
## 3. Problem Feasibility

| Requirement | How it is covered in this notebook |
|---|---|
| Sufficient data available | Cell 3.1: rows, positives and positives-per-feature |
| Reliable target labels exist | Cell 3.2: label consistency with exit dates |
| Data quality and representativeness | Cell 3.2: source mix, time span, coverage of monthly tables |
| Project cost, time and compute | Cell 3.3: measured training times |
| Is the expected prediction achievable | Cell 3.3: cross-validated AUC well above 0.5 but modest |
| ML better than a simple rule-based solution | Cell 3.3: rule vs logistic regression, same folds |

In [8]:
print(f'{len(raw):,} rows, {int(y_all.sum())} positives ({100 * y_all.mean():.1f}%), {len(C.NUMERIC_CORE) + len(C.CATEGORICAL_CORE)} candidate features')
print('events per candidate feature:', round(y_all.sum() / (len(C.NUMERIC_CORE) + len(C.CATEGORICAL_CORE)), 1), '(>10 is comfortable for regularised models)')

2,389 rows, 349 positives (14.6%), 31 candidate features
events per candidate feature: 11.3 (>10 is comfortable for regularised models)


In [9]:
left = raw[raw[C.TARGET] == 1]
days = (left.exit_date - left.observation_date).dt.days
print('label reliability: all leavers have an exit_date within 12 months of the snapshot ->', bool(days.between(0, 366).all()), '| range', int(days.min()), 'to', int(days.max()), 'days')
print('stayers with an exit_date:', int(raw.loc[raw[C.TARGET] == 0, 'exit_date'].notna().sum()))
print(raw.groupby('source')[C.TARGET].agg(['count', 'mean']).round(3))
print('representativeness caveat: 2,000 historical rows (2021-2025) have no attendance/payroll history; only the 30 Sep 2025 snapshot rows do')

label reliability: all leavers have an exit_date within 12 months of the snapshot -> True | range 3 to 360 days
stayers with an exit_date: 0
                          count   mean
source                                
historical_extension       2000  0.150
master_employee_snapshot    389  0.129
representativeness caveat: 2,000 historical rows (2021-2025) have no attendance/payroll history; only the 30 Sep 2025 snapshot rows do


In [10]:
S = runs.load_model_frame(True); Xtr, ytr, Xte, yte, folds, dfj = S['Xtr'], S['ytr'], S['Xte'], S['yte'], S['folds'], S['df']
mk = lambda est, monthly=True: Pipeline([('features', F.build_pipeline(monthly)), ('model', est)])
LRB = lambda **kw: LogisticRegression(class_weight='balanced', max_iter=2000, random_state=SEED, **kw)
t0 = time.time(); oof_lr = cross_val_predict(mk(LRB()), Xtr, ytr, cv=folds, method='predict_proba')[:, 1]; train_s = time.time() - t0
rule_oof = Xtr.months_since_last_raise.fillna(Xtr.months_since_last_raise.median()).to_numpy()
print(f'ROC-AUC  rule: {roc_auc_score(ytr, rule_oof):.3f}   logistic regression (5-fold OOF): {roc_auc_score(ytr, oof_lr):.3f}   -> ML beats the single rule, but only moderately')
print(f'compute: 5-fold fit + predict took {train_s:.1f}s on a laptop CPU: cost and time are small')
print('feasible: yes - signal exists (AUC about 0.7) but the problem is hard; plan for decision SUPPORT, not decision making')

ROC-AUC  rule: 0.598   logistic regression (5-fold OOF): 0.710   -> ML beats the single rule, but only moderately
compute: 5-fold fit + predict took 1.2s on a laptop CPU: cost and time are small
feasible: yes - signal exists (AUC about 0.7) but the problem is hard; plan for decision SUPPORT, not decision making


---
---
# PHASE 2: DATA COLLECTION AND UNDERSTANDING

---
## 4. Data Collection

| Requirement | How it is covered in this notebook |
|---|---|
| Collect from CSV/Excel/SQL/APIs/warehouses | Cell 4.1: Excel workbook (26 sheets) and its CSV exports; other source types noted |
| Data owners and sources; permissions and privacy | Cell 4.2: README sheet (fictional company, who owns what); protected attributes excluded |
| Collection methods, sampling | Cell 4.2: snapshot rows + historical extension rows |
| Freshness, historical coverage, update frequency | Cell 4.3: date ranges per table |
| Verify schema and data dictionary; meaning and units of columns | Cell 4.4: columns vs `data_dictionary` sheet |
| Represents intended population; target defined consistently | Cell 4.5 |

In [11]:
xl = pd.ExcelFile(C.data_path()); print(len(xl.sheet_names), 'sheets:', xl.sheet_names)
print('CSV exports in 03_Data/csv:', len(list((ROOT / '03_Data' / 'csv').glob('*.csv'))), '| other sources (SQL, APIs, warehouses) are not used in this phase')

26 sheets: ['README', 'data_dictionary', 'attrition_dataset', 'attrition_scoring_2026', 'engagement_survey', 'performance_reviews', 'attendance_monthly_features', 'attendance_daily_examples', 'payroll_monthly_features', 'monthly_headcount_history', 'workforce_events', 'salary_training', 'job_descriptions', 'resumes', 'resume_job_labels', 'resume_bias_pairs', 'chatbot_nlu_training', 'chatbot_test_set', 'chatbot_responses', 'hr_faq', 'chatbot_dialogues', 'baseline_benchmarks', 'anomaly_baselines', 'sample_attrition_risk', 'sample_anomaly_results', 'sample_forecast']
CSV exports in 03_Data/csv: 26 | other sources (SQL, APIs, warehouses) are not used in this phase


In [12]:
readme = H.load('README'); readme.dropna(how='all').head(14)

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,CortexHR AI/ML Dataset,NaN,NaN,NaN,NaN
1,NaN,Aura Workforce · CortexHR (AI/ML Developers track) | Zenith Nova Technologies (fictional) ...,NaN,NaN,NaN,NaN
3,NaN,About this dataset,NaN,NaN,NaN,NaN
5,NaN,"Everything in this file is FICTIONAL. The company ""Zenith Nova Technologies Pvt Ltd"", its people...",NaN,NaN,NaN,NaN
7,NaN,This workbook is complete on its own. It comes from the Zenith Nova master dataset that all six ...,NaN,NaN,NaN,NaN
9,NaN,What is in this workbook,NaN,NaN,NaN,NaN
11,NaN,"A complete practice kit for the AI/ML team: a leakage-trap attrition table, anomaly-detection se...",NaN,NaN,NaN,NaN
13,NaN,Headline numbers,NaN,NaN,NaN,NaN
15,NaN,Item,NaN,Value,NaN,NaN
16,NaN,Attrition table,NaN,"2,389 rows = 383 real employee snapshots (taken 30 Sep 2025) + 2,000 historical extension rows; ...",NaN,NaN


In [13]:
att, pay = D.load_sheet(C.SHEET_ATT), D.load_sheet(C.SHEET_PAY)
pd.DataFrame({'table': ['attrition (observation_date)', 'attendance (month)', 'payroll (month)'],
              'from': [raw.observation_date.min().date(), att.month.min().date(), pay.month.min().date()],
              'to': [raw.observation_date.max().date(), att.month.max().date(), pay.month.max().date()],
              'rows': [len(raw), len(att), len(pay)], 'update frequency': ['snapshots', 'monthly', 'monthly']})

,table,from,to,rows,update frequency
0,attrition (observation_date),2021-01-01,2025-09-30,2389,snapshots
1,attendance (month),2025-04-01,2026-09-01,7339,monthly
2,payroll (month),2025-04-01,2026-09-01,7355,monthly


In [14]:
dd = H.load('data_dictionary'); tcol, ccol = dd.columns[0], dd.columns[1]
d_attr = dd[dd[tcol].astype(str).str.contains('attrition_dataset', na=False)]
missing_in_dict = sorted(set(raw.columns) - set(d_attr[ccol].astype(str)))
print('columns documented in data_dictionary:', len(d_attr), '| undocumented columns:', missing_in_dict)
d_attr.head(8)

columns documented in data_dictionary: 42 | undocumented columns: []


,sheet,column,data_type,description,null_count,null_percent,distinct_values,example
0,attrition_dataset,row_id,integer,Row number in the sheet.,0,0.0,2389,1
1,attrition_dataset,employee_id,text,Unique employee code (EMP-0001 ...). Same code in every table and every workbook.,0,0.0,2383,EMP-0002
2,attrition_dataset,observation_date,date,Date the features were measured (30 Sep 2025 for real snapshots).,0,0.0,1135,2025-09-30
3,attrition_dataset,department_id,text,Department code (D00 Executive ... D10 Legal & Compliance).,0,0.0,10,D08
4,attrition_dataset,role_id,text,Job role code from the roles table.,0,0.0,48,R047
5,attrition_dataset,job_level,integer,"Level 1 to 7 (1 Associate ... 6 Head, 7 CEO).",0,0.0,6,4
6,attrition_dataset,work_location,text,City of the office the employee is attached to.,0,0.0,16,Pune
7,attrition_dataset,work_mode,text,"Onsite, Hybrid or Remote.",0,0.0,3,Remote


In [15]:
print('target defined consistently: left_within_12m == (exit within 365 days of observation_date):',
      bool(((raw.exit_date - raw.observation_date).dt.days.between(0, 366) == (raw[C.TARGET] == 1)).all()))
print('units: money in INR per year; tenure in months; rates are fractions 0-1; hours per day')

target defined consistently: left_within_12m == (exit within 365 days of observation_date): True
units: money in INR per year; tenure in months; rates are fractions 0-1; hours per day


---
## 5. Data Loading and Initial Inspection

| Requirement | How it is covered in this notebook |
|---|---|
| Load dataset; first and last records; rows and columns | Cells 5.1-5.2 |
| Column names and data types; numeric/categorical/datetime/text/id columns | Cell 5.3 |
| Descriptive statistics; unique values per feature | Cell 5.4 |
| Constant and near-constant columns | Cell 5.5 |
| Dataset not empty / correctly loaded; expected schema and integrity | Cell 5.6 |

In [16]:
print(raw.shape); raw.head(3)

(2389, 42)


,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,tenure_months,total_experience_years,previous_employers_count,commute_distance_km,annual_fixed_pay,comp_ratio,raises_received,last_raise_pct,months_since_last_raise,promotions_received,months_since_last_promotion,internal_transfers,last_performance_rating,last_engagement_score,manager_changes_12m,training_hours_12m,attendance_rate_3m,late_arrivals_3m,avg_overtime_hours_per_month_3m,avg_hours_worked_3m,wfh_share_3m,unpaid_leave_days_6m,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
0,1,EMP-0002,2025-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,164,30.8,1,12.2,1733520,0.989,6,6.3,5,0,NaN,1,3.0,2.5,0,34.0,0.9219,3,0.00,8.01,0.891,2.0,5,0.2353,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
1,2,EMP-0004,2025-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,162,17.5,1,NaN,1247256,0.854,6,9.6,5,0,NaN,0,4.0,2.4,0,48.1,0.9688,1,0.00,8.35,0.469,1.0,4,0.0556,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
2,3,EMP-0011,2025-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,154,12.8,0,44.5,288492,0.872,7,9.5,5,0,NaN,0,4.0,2.7,0,57.7,0.9453,0,4.33,8.23,0.297,1.0,7,0.0000,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN


In [17]:
raw.tail(3)

,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,tenure_months,total_experience_years,previous_employers_count,commute_distance_km,annual_fixed_pay,comp_ratio,raises_received,last_raise_pct,months_since_last_raise,promotions_received,months_since_last_promotion,internal_transfers,last_performance_rating,last_engagement_score,manager_changes_12m,training_hours_12m,attendance_rate_3m,late_arrivals_3m,avg_overtime_hours_per_month_3m,avg_hours_worked_3m,wfh_share_3m,unpaid_leave_days_6m,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
2386,2387,EMP-0060,2025-09-30,D01,R006,3,Bengaluru,Hybrid,Permanent,MCA,111,17.6,2,8.8,1464708,1.003,6,8.1,5,0,NaN,0,3.0,3.1,0,16.2,1.0000,3,0.00,8.65,0.438,0.0,1,0.1448,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
2387,2388,EMP-0237,2025-09-30,D07,R040,2,Chennai,Remote,Permanent,B.Sc,51,9.4,1,7.5,486732,1.000,4,2.0,5,0,NaN,1,2.0,3.5,0,38.5,0.9844,1,11.67,9.07,0.922,0.0,9,0.0000,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN
2388,2389,EMP-0570,2025-09-30,D04,R024,3,Bengaluru,Hybrid,Permanent,B.E. Mechanical,10,9.9,2,11.9,1291032,1.020,0,NaN,10,0,NaN,0,NaN,4.0,0,12.9,0.9375,3,0.00,8.47,0.391,1.0,5,0.2051,0,0,master_employee_snapshot,NaT,NaT,NaN,NaN,NaN


In [18]:
kinds = {c: ('identifier' if c in C.ID_COLS else 'datetime' if 'date' in c else 'text' if c == 'exit_reason' else 'categorical' if raw[c].dtype == object or str(raw[c].dtype).startswith('str') else 'numeric') for c in raw.columns}
print(pd.Series(kinds).value_counts().to_dict()); pd.DataFrame({'dtype': raw.dtypes.astype(str), 'kind': pd.Series(kinds)}).T

{'numeric': 28, 'categorical': 8, 'datetime': 3, 'identifier': 2, 'text': 1}


,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,tenure_months,total_experience_years,previous_employers_count,commute_distance_km,annual_fixed_pay,comp_ratio,raises_received,last_raise_pct,months_since_last_raise,promotions_received,months_since_last_promotion,internal_transfers,last_performance_rating,last_engagement_score,manager_changes_12m,training_hours_12m,attendance_rate_3m,late_arrivals_3m,avg_overtime_hours_per_month_3m,avg_hours_worked_3m,wfh_share_3m,unpaid_leave_days_6m,paid_leave_days_6m,dept_attrition_rate_prior_12m,left_within_12m,voluntary_left_within_12m,source,exit_date,resignation_date,exit_reason,exit_interview_done,fnf_settlement_amount
dtype,int64,str,datetime64[us],str,str,int64,str,str,str,str,int64,float64,int64,float64,int64,float64,int64,float64,int64,int64,float64,int64,float64,float64,int64,float64,float64,int64,float64,float64,float64,float64,int64,float64,int64,int64,str,datetime64[us],datetime64[us],str,str,float64
kind,identifier,identifier,datetime,categorical,categorical,numeric,categorical,categorical,categorical,categorical,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,numeric,categorical,datetime,datetime,text,categorical,numeric


In [19]:
raw.describe(include='all').T[['count', 'unique', 'mean', 'std', 'min', 'max']].head(40)

,count,unique,mean,std,min,max
row_id,2389.0,NaN,1195.0,689.789219,1.0,2389.0
employee_id,2389,2383,NaN,NaN,NaN,NaN
observation_date,2389,NaN,2023-07-02 11:23:31.971536,NaN,2021-01-01 00:00:00,2025-09-30 00:00:00
department_id,2389,10,NaN,NaN,NaN,NaN
role_id,2389,48,NaN,NaN,NaN,NaN
job_level,2389.0,NaN,2.491419,1.134039,1.0,6.0
work_location,2389,16,NaN,NaN,NaN,NaN
work_mode,2389,3,NaN,NaN,NaN,NaN
employment_type,2389,2,NaN,NaN,NaN,NaN
highest_education,2374,27,NaN,NaN,NaN,NaN


In [20]:
nun = raw.nunique().sort_values(); top = raw.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0])
print('constant columns:', list(nun[nun <= 1].index), '| near-constant (>=98% one value):', list(top[top >= .98].index))
pd.DataFrame({'n_unique': nun, 'top_value_share': top.round(3)}).head(12)

constant columns: [] | near-constant (>=98% one value): []


,n_unique,top_value_share
annual_fixed_pay,2364,0.001
attendance_rate_3m,1051,0.033
avg_hours_worked_3m,181,0.019
avg_overtime_hours_per_month_3m,390,0.813
commute_distance_km,372,0.023
comp_ratio,585,0.006
department_id,10,0.338
dept_attrition_rate_prior_12m,1082,0.062
employee_id,2383,0.001
employment_type,2,0.937


In [21]:
assert len(raw) > 0 and raw.shape == (2389, 42) and not raw.columns.duplicated().any()
print('integrity: non-empty, 2,389 x 42, unique column names, row_id unique ->', raw.row_id.is_unique)

integrity: non-empty, 2,389 x 42, unique column names, row_id unique -> True


---
## 6. Exploratory Data Analysis (EDA)

| Requirement | How it is covered in this notebook |
|---|---|
| Univariate analysis | Cells 6.1-6.2 |
| Bivariate analysis | Cell 6.4 |
| Multivariate analysis | Cells 6.5, 6.7 |
| Numerical distributions, central tendency, dispersion | Cell 6.1 (histograms + mean/median/std/skew) |
| Categorical frequency distributions | Cell 6.2 |
| Target variable distribution; features vs target | Cells 6.3-6.4 |
| Correlations and multicollinearity | Cell 6.6 (correlation pairs + VIF) |
| Patterns, trends, clusters, interactions | Cell 6.7 (interaction table + KMeans clusters) |
| Outliers and unusual observations | Cell 6.8 |
| Subgroup distributions | Cell 6.9 |
| Date ranges, seasonality, time patterns | Cell 6.10 |
| Potential data-quality problems | Cell 6.11 |
| Form hypotheses and validate them | Cell 6.12 (statistical tests) |

In [22]:
num_show = ['tenure_months', 'annual_fixed_pay', 'comp_ratio', 'months_since_last_raise', 'last_engagement_score', 'avg_overtime_hours_per_month_3m', 'commute_distance_km', 'training_hours_12m', 'late_arrivals_3m']
fig, ax = plt.subplots(3, 3, figsize=(13, 8))
for a, c in zip(ax.ravel(), num_show): a.hist(raw[c].dropna(), bins=40); a.set_title(c, fontsize=9)
fig.tight_layout()
raw[num_show].agg(['mean', 'median', 'std', 'skew']).T.round(2)

,mean,median,std,skew
tenure_months,37.78,31.00,27.43,1.40
annual_fixed_pay,1029199.02,757488.00,856880.81,4.80
comp_ratio,1.03,1.03,0.14,0.09
months_since_last_raise,10.99,9.00,8.01,1.37
last_engagement_score,3.44,3.50,0.75,-0.07
avg_overtime_hours_per_month_3m,2.17,0.00,5.33,2.97
commute_distance_km,12.41,9.70,9.36,2.03
training_hours_12m,26.32,21.70,17.43,2.10
late_arrivals_3m,4.97,4.00,3.93,1.60


In [23]:
for c in ['department_id', 'work_mode', 'employment_type', 'work_location']:
    print(c, raw[c].value_counts(dropna=False).head(8).to_dict())
raw.highest_education.value_counts(dropna=False).head(6)

department_id {'D01': 808, 'D06': 323, 'D04': 267, 'D03': 244, 'D09': 229, 'D02': 161, 'D08': 111, 'D05': 102}
work_mode {'Hybrid': 1107, 'Onsite': 829, 'Remote': 453}
employment_type {'Permanent': 2239, 'Fixed-Term Contract': 150}
work_location {'Bengaluru': 888, 'Chennai': 443, 'Hyderabad': 292, 'Pune': 228, 'Mumbai': 155, 'Delhi NCR': 137, 'Coimbatore': 129, 'Kochi': 53}


highest_education
B.Tech Information Technology    244
M.Tech                           233
B.E. Computer Science            229
MCA                              210
MBA                              198
B.Com                            178
Name: count, dtype: int64

In [24]:
vc = raw[C.TARGET].value_counts(); print(vc.to_dict(), f'-> {100 * y_all.mean():.2f}% left'); A.plot_class_balance(A.class_balance(raw));

{0: 2040, 1: 349} -> 14.61% left


In [25]:
W.plot_eda(raw);

In [26]:
rows = []
for c in ['tenure_months', 'comp_ratio', 'months_since_last_raise', 'last_engagement_score', 'last_performance_rating', 'avg_overtime_hours_per_month_3m', 'commute_distance_km', 'annual_fixed_pay']:
    a, b = raw.loc[y_all == 1, c].dropna(), raw.loc[y_all == 0, c].dropna()
    rows.append({'feature': c, 'mean_left': a.mean(), 'mean_stayed': b.mean(), 'mann_whitney_p': st.mannwhitneyu(a, b).pvalue})
pd.DataFrame(rows).round(4).sort_values('mann_whitney_p')

,feature,mean_left,mean_stayed,mann_whitney_p
3,last_engagement_score,3.1356,3.492100e+00,0.0000
2,months_since_last_raise,13.9484,1.048380e+01,0.0000
1,comp_ratio,1.0095,1.036700e+00,0.0015
6,commute_distance_km,14.4238,1.206640e+01,0.0062
4,last_performance_rating,3.1931,3.341500e+00,0.0087
0,tenure_months,33.8138,3.845440e+01,0.0168
5,avg_overtime_hours_per_month_3m,3.1437,2.001900e+00,0.0298
7,annual_fixed_pay,964837.1347,1.040210e+06,0.0986


In [27]:
corr = raw[C.NUMERIC_CORE].corr(); fig, ax = plt.subplots(figsize=(9, 7)); im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=6); ax.set_yticks(range(len(corr)), corr.columns, fontsize=6); fig.colorbar(im); ax.set_title('Correlation of numeric inputs')

Text(0.5, 1.0, 'Correlation of numeric inputs')

In [28]:
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().rename('r').reset_index(); pairs['abs_r'] = pairs.r.abs()
print('most correlated input pairs:'); print(pairs.sort_values('abs_r', ascending=False).head(6).round(3).to_string(index=False))
Z = raw[C.NUMERIC_CORE].fillna(raw[C.NUMERIC_CORE].median()); Z = (Z - Z.mean()) / Z.std()
vif = {c: 1 / (1 - np.corrcoef(Z[c], np.linalg.lstsq(Z.drop(columns=c).to_numpy(), Z[c].to_numpy(), rcond=None)[0] @ Z.drop(columns=c).to_numpy().T)[0, 1] ** 2) for c in Z.columns}
print('\nVIF (multicollinearity; >5-10 is a warning):'); pd.Series(vif).sort_values(ascending=False).head(6).round(2)

most correlated input pairs:
               level_0                     level_1     r  abs_r
         tenure_months             raises_received 0.918  0.918
             job_level      total_experience_years 0.808  0.808
             job_level            annual_fixed_pay 0.764  0.764
         tenure_months months_since_last_promotion 0.667  0.667
total_experience_years            annual_fixed_pay 0.632  0.632
       raises_received months_since_last_promotion 0.593  0.593

VIF (multicollinearity; >5-10 is a warning):


tenure_months                  8.97
total_experience_years         8.23
job_level                      7.95
raises_received                6.51
annual_fixed_pay               2.79
months_since_last_promotion    1.43
dtype: float64

In [29]:
tb = pd.cut(raw.tenure_months.where(raw.tenure_months >= 0), [0, 12, 24, 48, 400], labels=['<1y', '1-2y', '2-4y', '4y+'])
print('interaction: leaver rate (%) by tenure band x work mode'); print((100 * raw.pivot_table(index=tb, columns='work_mode', values=C.TARGET, aggfunc='mean', observed=True)).round(1))
from sklearn.cluster import KMeans
Zc = raw[['tenure_months', 'comp_ratio', 'months_since_last_raise', 'last_engagement_score', 'avg_overtime_hours_per_month_3m']].fillna(raw.median(numeric_only=True)); Zc = (Zc - Zc.mean()) / Zc.std()
km = KMeans(4, n_init=10, random_state=SEED).fit(Zc); print('\nKMeans employee clusters: leaver rate %'); print((100 * raw.groupby(km.labels_)[C.TARGET].mean()).round(1).to_dict())

interaction: leaver rate (%) by tenure band x work mode
work_mode      Hybrid  Onsite  Remote
tenure_months                        
<1y               9.8    20.2     5.8
1-2y             18.0    19.6    15.0
2-4y             12.8    24.1    15.2
4y+               8.1    11.6     9.9

KMeans employee clusters: leaver rate %
{0: 10.4, 1: 22.0, 2: 20.9, 3: 13.2}


In [30]:
pd.DataFrame({'IQR outliers': ((raw[num_show] < raw[num_show].quantile(.25) - 1.5 * (raw[num_show].quantile(.75) - raw[num_show].quantile(.25))) | (raw[num_show] > raw[num_show].quantile(.75) + 1.5 * (raw[num_show].quantile(.75) - raw[num_show].quantile(.25)))).sum()})

,IQR outliers
tenure_months,74
annual_fixed_pay,94
comp_ratio,33
months_since_last_raise,73
last_engagement_score,0
avg_overtime_hours_per_month_3m,446
commute_distance_km,141
training_hours_12m,99
late_arrivals_3m,67


In [31]:
raw.groupby('department_id').agg(employees=('row_id', 'size'), leaver_rate_pct=(C.TARGET, lambda s: round(100 * s.mean(), 1)), median_pay=('annual_fixed_pay', 'median'), median_engagement=('last_engagement_score', 'median')).round(1)

,employees,leaver_rate_pct,median_pay,median_engagement
department_id,,,,
D01,808,13.6,936294.0,3.4
D02,161,14.3,958416.0,3.5
D03,244,13.1,948990.0,3.5
D04,267,13.9,877584.0,3.5
D05,102,11.8,767934.0,3.4
D06,323,21.1,326040.0,3.4
D07,82,11.0,486732.0,3.5
D08,111,13.5,530532.0,3.4
D09,229,17.0,487404.0,3.5


In [32]:
by = raw.assign(year=raw.observation_date.dt.year, month=raw.observation_date.dt.month)
print('rows and leaver rate by observation year'); print(by.groupby('year')[C.TARGET].agg(['size', 'mean']).round(3))
print('\nleaver rate by observation month'); print(by.groupby('month')[C.TARGET].mean().round(3).to_dict())

rows and leaver rate by observation year
      size   mean
year             
2021   514  0.150
2022   467  0.135
2023   446  0.143
2024   471  0.163
2025   491  0.138

leaver rate by observation month
{1: 0.15, 2: 0.149, 3: 0.187, 4: 0.134, 5: 0.135, 6: 0.188, 7: 0.166, 8: 0.189, 9: 0.137, 10: 0.138, 11: 0.096, 12: 0.103}


In [33]:
audit_t, lk, single = A.build_audit_table(raw, att, pay); print(len(audit_t), 'data-quality problems found (details in Section 10 and reports/data_audit_report.md)'); audit_t[['problem', 'rows_affected', 'severity']]

17 data-quality problems found (details in Section 10 and reports/data_audit_report.md)


,problem,rows_affected,severity
0,Post-outcome (leakage) columns,349,Critical
1,Exact duplicate snapshots,6,High
2,Impossible value: negative tenure,4,High
3,Impossible value: negative total experience,10,High
4,Impossible value: experience shorter than tenure,140,High
5,Impossible value: attendance_rate_3m above 100%,2,Medium
6,Impossible value: wfh_share_3m above 100%,23,Medium
7,Impossible value: wfh_share_3m below 0%,190,Medium
8,Inconsistent city spellings,13,Medium
9,Missing values in predictors,1459,Medium


In [34]:
tests = []
stale = raw.months_since_last_raise > 18; ct = pd.crosstab(stale, y_all); tests.append(('H1 raise older than 18 months -> more leavers', 100 * y_all[stale].mean(), 100 * y_all[~stale].mean(), st.chi2_contingency(ct)[1]))
low = raw.last_engagement_score < 3; ct = pd.crosstab(low, y_all); tests.append(('H2 engagement below 3 -> more leavers', 100 * y_all[low].mean(), 100 * y_all[~low].mean(), st.chi2_contingency(ct)[1]))
npm = (raw.promotions_received == 0) & (raw.tenure_months > 36); ct = pd.crosstab(npm, y_all); tests.append(('H3 never promoted after 3 years -> more leavers', 100 * y_all[npm].mean(), 100 * y_all[~npm].mean(), st.chi2_contingency(ct)[1]))
pd.DataFrame(tests, columns=['hypothesis', 'leaver % in group', 'leaver % elsewhere', 'chi2 p-value']).round(4)

,hypothesis,leaver % in group,leaver % elsewhere,chi2 p-value
0,H1 raise older than 18 months -> more leavers,30.6494,11.5269,0.0000
1,H2 engagement below 3 -> more leavers,18.4426,13.6244,0.0089
2,H3 never promoted after 3 years -> more leavers,11.5473,15.2863,0.0551


---
---
# PHASE 3: DATA QUALITY AND DATA CLEANING

---
## 7. Missing-Value Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Missing values per column; counts and percentages | Cell 7.1 |
| Why values are missing; systematic or informative? | Cell 7.2 |
| Remove records/columns only when justified | Cell 7.3 decision table |
| Impute numerical values (strategies compared) | Cell 7.4 |
| Impute categorical values | Cell 7.3 / pipeline `SimpleImputer(constant 'Unknown')` |
| Missingness indicators | Cell 7.4 compares with/without indicators |
| Missing target labels handled separately | Cell 7.1 (target has none) |
| Avoid arbitrary replacements that distort meaning | Cell 7.3: never-promoted blanks are NOT filled with 0 months |

In [35]:
ms = raw.isna().sum().to_frame('missing').assign(pct=lambda d: (100 * d.missing / len(raw)).round(2)); ms = ms[ms.missing > 0].sort_values('pct', ascending=False)
print('missing target labels:', int(raw[C.TARGET].isna().sum())); ms

missing target labels: 0


,missing,pct
resignation_date,2068,86.56
exit_date,2040,85.39
exit_reason,2040,85.39
fnf_settlement_amount,2040,85.39
exit_interview_done,2040,85.39
months_since_last_promotion,1459,61.07
last_engagement_score,528,22.10
last_raise_pct,202,8.46
last_performance_rating,182,7.62
commute_distance_km,55,2.30


In [36]:
rows = []
for c in ms.index.difference(C.LEAKAGE_CONFIRMED):
    m = raw[c].isna(); rows.append({'column': c, 'missing': int(m.sum()), 'leaver % when missing': round(100 * y_all[m].mean(), 1), 'leaver % when present': round(100 * y_all[~m].mean(), 1)})
print('is missingness informative? (exit columns excluded: blank by design for stayers)'); pd.DataFrame(rows)

is missingness informative? (exit columns excluded: blank by design for stayers)


,column,missing,leaver % when missing,leaver % when present
0,commute_distance_km,55,9.1,14.7
1,highest_education,15,13.3,14.6
2,last_engagement_score,528,16.7,14.0
3,last_performance_rating,182,15.4,14.5
4,last_raise_pct,202,16.3,14.4
5,months_since_last_promotion,1459,15.1,13.8


In [37]:
pd.DataFrame([('months_since_last_promotion', 'structural: blank = never promoted', 'derive never_promoted flag; impute number; do NOT write 0'),
              ('last_engagement_score', 'no survey reply (possibly informative)', 'median + missing indicator'),
              ('last_performance_rating / last_raise_pct / months_since_last_raise', 'no review/raise on record', 'median + missing indicator'),
              ('commute_distance_km, highest_education', 'accidental blanks (data entry)', 'median / most frequent'),
              ('exit_* and fnf columns', 'blank for stayers by design', 'DROP: leakage, not a missing-value problem')],
             columns=['column', 'why missing', 'treatment'])

,column,why missing,treatment
0,months_since_last_promotion,structural: blank = never promoted,derive never_promoted flag; impute number; do NOT write 0
1,last_engagement_score,no survey reply (possibly informative),median + missing indicator
2,last_performance_rating / last_raise_pct / months_since_last_raise,no review/raise on record,median + missing indicator
3,"commute_distance_km, highest_education",accidental blanks (data entry),median / most frequent
4,exit_* and fnf columns,blank for stayers by design,"DROP: leakage, not a missing-value problem"


In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
Xc = D.clean_raw(Xtr); ncols = [c for c in C.NUMERIC_CORE if c != 'annual_fixed_pay']
def quick_auc(imputer, scaler=None):
    p = Pipeline([('imp', imputer), ('sc', scaler or StandardScaler()), ('lr', LRB())]); return roc_auc_score(ytr, cross_val_predict(p, Xc[ncols], ytr, cv=folds, method='predict_proba')[:, 1])
print(pd.Series({'median': quick_auc(SimpleImputer(strategy='median')), 'mean': quick_auc(SimpleImputer(strategy='mean')),
                 'median + missing indicators': quick_auc(SimpleImputer(strategy='median', add_indicator=True)), 'KNN (k=5)': quick_auc(KNNImputer(n_neighbors=5))}, name='5-fold OOF ROC-AUC').round(4))

median                         0.7049
mean                           0.7041
median + missing indicators    0.7017
KNN (k=5)                      0.6994
Name: 5-fold OOF ROC-AUC, dtype: float64


---
## 8. Duplicate-Record Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Exact duplicate rows | Cell 8.1 |
| Duplicate business keys / entity records | Cell 8.1 |
| Erroneous or legitimate repeated events? | Cell 8.2 |
| Remove only confirmed duplicates | Cell 8.3 |
| Prevent duplicates leaking across splits | Cell 8.4 (employee overlap between train and test = 0) |

In [39]:
body = raw.drop(columns=['row_id'])
print('exact duplicate rows:', int(body.duplicated().sum()), '| duplicate (employee_id, observation_date):', int(raw.duplicated(['employee_id', 'observation_date']).sum()), '| repeated employee_id:', int(raw.employee_id.duplicated().sum()))
feat_cols = [c for c in raw.columns if c not in C.ID_COLS + [C.TARGET] + C.LEAKAGE_CONFIRMED]
print('near-duplicates (identical on all feature columns, different ids):', int(raw.duplicated(feat_cols).sum()))

exact duplicate rows: 6 | duplicate (employee_id, observation_date): 6 | repeated employee_id: 6
near-duplicates (identical on all feature columns, different ids): 6


In [40]:
d = raw[raw.employee_id.duplicated(keep=False)].sort_values('employee_id')
same = d.drop(columns='row_id').groupby('employee_id').nunique(dropna=False).max(axis=1)
print('each repeated employee has identical values in every column (nunique == 1):', bool((same == 1).all()), '-> ERRONEOUS duplicates (not repeated events: same observation_date, same values)')

each repeated employee has identical values in every column (nunique == 1): True -> ERRONEOUS duplicates (not repeated events: same observation_date, same values)


In [41]:
dedup, dlog = D.prepare_attrition(raw); print(dlog)

{'rows_before': 2389, 'exact_duplicates_dropped': 6, 'rows_after': 2383}


In [42]:
ids_tr, ids_te = set(dfj.loc[Xtr.index, 'employee_id']), set(dfj.loc[Xte.index, 'employee_id'])
print('employee overlap between train and test:', len(ids_tr & ids_te), '(must be 0)'); assert not (ids_tr & ids_te)

employee overlap between train and test: 0 (must be 0)


---
## 9. Outlier Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Unusual numerical observations; statistical methods, visualisations, domain knowledge | Cells 9.1-9.2 |
| Genuine extremes vs data-entry errors | Cell 9.3 (pay typos vs senior pay) |
| Correct invalid values when the correct value is known | Cell 9.3 |
| Transformation, capping or robust methods | Cell 9.4 |
| Avoid deleting all outliers | Cell 9.5 |

In [43]:
num = raw.select_dtypes('number').drop(columns=['row_id', C.TARGET, 'voluntary_left_within_12m'])
q1, q3 = num.quantile(.25), num.quantile(.75); iqr = q3 - q1
t = pd.DataFrame({'iqr': ((num < q1 - 1.5 * iqr) | (num > q3 + 1.5 * iqr)).sum(), 'z>3': ((num - num.mean()) / num.std(ddof=0)).abs().gt(3).sum()}).sort_values('iqr', ascending=False); t.head(8)

,iqr,z>3
avg_overtime_hours_per_month_3m,446,61
internal_transfers,189,189
commute_distance_km,141,47
promotions_received,138,50
job_level,126,2
training_hours_12m,99,47
annual_fixed_pay,94,26
dept_attrition_rate_prior_12m,87,40


In [44]:
R.show_figure('outliers_boxplot', 10);

In [45]:
typo = A.pay_consistency_outliers(raw); print('pay typo candidates (pay / comp_ratio is ~11x the role mid-point):'); print(typo.to_string(index=False))
print('\nsenior pay is real: top-3 pay of ordinary rows vs typo rows ->', raw.annual_fixed_pay.drop(typo.index).nlargest(3).round(0).tolist(), 'vs', typo.annual_fixed_pay.round(0).tolist())

pay typo candidates (pay / comp_ratio is ~11x the role mid-point):
 row_id employee_id role_id  annual_fixed_pay  comp_ratio  ratio_to_role_midpoint
     15    EMP-0034    R015           4888200       0.966                   11.37
    217    EMP-0433    R006          18671760       1.279                   11.11
    345    EMP-0601    R034           3606000       0.975                   11.11

senior pay is real: top-3 pay of ordinary rows vs typo rows -> [5649372, 4642920, 4576800] vs [4888200, 18671760, 3606000]


In [46]:
d_ = Xtr.copy(); d_['annual_fixed_pay'] = d_.annual_fixed_pay.where(~d_.index.isin(typo.index))
print('skew of pay:', round(d_.annual_fixed_pay.skew(), 2), '-> after log1p:', round(np.log1p(d_.annual_fixed_pay).skew(), 2))
lo, hi = d_.annual_fixed_pay.quantile([.01, .99]); print('winsorising at 1%/99%% would change', int(((d_.annual_fixed_pay < lo) | (d_.annual_fixed_pay > hi)).sum()), 'values')
print(pd.Series({'StandardScaler': quick_auc(SimpleImputer(strategy='median'), StandardScaler()), 'RobustScaler': quick_auc(SimpleImputer(strategy='median'), RobustScaler())}, name='OOF ROC-AUC').round(4).to_dict())

skew of pay: 1.54 -> after log1p: 0.12
winsorising at 1%/99%% would change 40 values
{'StandardScaler': 0.7049, 'RobustScaler': 0.705}


**Decision.** Pay typos are blanked inside the pipeline using *training* role mid-points and imputed; pay is log-transformed; scalers are compared; genuine extremes (senior pay, many transfers) are **kept**, not deleted.

---
## 10. Data Consistency and Validity

| Requirement | How it is covered in this notebook |
|---|---|
| Standardize inconsistent labels; whitespace and formatting | Cells 10.1-10.2 |
| Validate dates, units, currencies, ranges; invalid/infinite/impossible values | Cell 10.3 |
| Data types and parsing; naming conventions | Cell 10.4 |
| Conflicting target labels | Cell 10.5 |
| Validate joins and relationships; referential integrity | Cell 10.6 |
| Data-entry and collection errors | Cell 10.3 + planted-issue check in Task 1 notebook |
| Audit trail of cleaning decisions | Cell 10.7 (saved to reports/tables/cleaning_audit_trail.csv) |

In [47]:
print('city spellings before:', raw.work_location.nunique(), '-> after:', D.canonical_city(raw.work_location).nunique()); raw.work_location.value_counts().head(16)

city spellings before: 16 -> after: 9


work_location
Bengaluru     888
Chennai       443
Hyderabad     292
Pune          228
Mumbai        155
Delhi NCR     137
Coimbatore    129
Kochi          53
Erode          51
BENGALURU       3
Bangalore       3
bengaluru       2
chennai         2
delhi ncr       1
Delhi-NCR       1
Madras          1
Name: count, dtype: int64

In [48]:
sc = raw.select_dtypes(exclude='number').select_dtypes(exclude='datetime'); ws = {c: int((sc[c].dropna().astype(str) != sc[c].dropna().astype(str).str.strip()).sum()) for c in sc.columns}
print('values with leading/trailing whitespace per text column:', {k: v for k, v in ws.items() if v} or 'none')

values with leading/trailing whitespace per text column: none


In [49]:
chk = {'exit_date before observation_date': int((raw.exit_date < raw.observation_date).sum()), 'attendance_rate_3m outside 0-1': int(((raw.attendance_rate_3m > 1) | (raw.attendance_rate_3m < 0)).sum()),
       'wfh_share_3m outside 0-1': int(((raw.wfh_share_3m > 1) | (raw.wfh_share_3m < 0)).sum()), 'rating/engagement outside 1-5': int(((raw.last_performance_rating.dropna() < 1) | (raw.last_performance_rating.dropna() > 5)).sum()),
       'negative tenure': int((raw.tenure_months < 0).sum()), 'negative or tiny experience': int((raw.total_experience_years < 0).sum()), 'experience < tenure': int((raw.total_experience_years < raw.tenure_months / 12 - .05).sum()),
       'non-positive pay': int((raw.annual_fixed_pay <= 0).sum()), 'infinite values': int(np.isinf(raw.select_dtypes('number')).sum().sum()), 'dates unparseable': int(raw.observation_date.isna().sum())}
pd.Series(chk, name='rows')

exit_date before observation_date      0
attendance_rate_3m outside 0-1         2
wfh_share_3m outside 0-1             213
rating/engagement outside 1-5          0
negative tenure                        4
negative or tiny experience           10
experience < tenure                  150
non-positive pay                       0
infinite values                        0
dates unparseable                      0
Name: rows, dtype: int64

In [50]:
print('dtype agreement CSV vs XLSX for numeric columns:', bool((raw_csv.select_dtypes('number').dtypes.astype(str) == raw.select_dtypes('number').dtypes.astype(str)).all()))
import re; bad = [c for c in raw.columns if not re.fullmatch(r'[a-z0-9_]+', c)]; print('column names not snake_case:', bad or 'none')

dtype agreement CSV vs XLSX for numeric columns: True
column names not snake_case: none


In [51]:
conf = raw.groupby('employee_id')[C.TARGET].nunique(); print('employees with conflicting target labels:', int((conf > 1).sum()))

employees with conflicting target labels: 0


In [52]:
ids_att, ids_raw = set(att.employee_id), set(raw.employee_id)
print('attendance employees missing from attrition table:', len(ids_att - ids_raw), '| payroll:', len(set(pay.employee_id) - ids_raw))
print('join report:', S['join_report'])

attendance employees missing from attrition table: 145 | payroll: 145
join report: {'rows_before_join': 2383, 'rows_after_join': 2383, 'cardinality': 'attrition(1) -> aggregates(1), validated one_to_one', 'attendance_matched_rows': 383, 'payroll_matched_rows': 383, 'unmatched_rows_attendance': 2000, 'unmatched_rows_payroll': 2000, 'rows_dropped': 0, 'window_months': 3, 'temporal_cutoff': 'only complete months with month_end <= observation_date; last `window` such months', 'payroll_exact_duplicates_removed': 15, 'rows_before': 2389, 'exact_duplicates_dropped': 6, 'rows_after': 2383}


In [53]:
trail = pd.DataFrame([('exact duplicates', 6, 'dropped (keep first)'), ('negative tenure', 4, 'set to missing, impute on train'), ('negative experience', 10, 'set to missing'),
    ('experience shorter than tenure', 140, 'set to missing'), ('attendance_rate_3m > 1', 2, 'clipped to 1'), ('wfh_share_3m outside 0-1', 213, 'clipped to 0-1'), ('city spellings', 13, 'canonical names'),
    ('salary typos', 3, 'blanked in pipeline using training role mid-point, then imputed'), ('payroll exact duplicates', 15, 'dropped before aggregation'), ('post-outcome columns', 6, 'excluded from every model input')],
    columns=['issue', 'rows', 'decision']); R.save_table(trail, 'cleaning_audit_trail'); trail

,issue,rows,decision
0,exact duplicates,6,dropped (keep first)
1,negative tenure,4,"set to missing, impute on train"
2,negative experience,10,set to missing
3,experience shorter than tenure,140,set to missing
4,attendance_rate_3m > 1,2,clipped to 1
5,wfh_share_3m outside 0-1,213,clipped to 0-1
6,city spellings,13,canonical names
7,salary typos,3,"blanked in pipeline using training role mid-point, then imputed"
8,payroll exact duplicates,15,dropped before aggregation
9,post-outcome columns,6,excluded from every model input


---
## 11. Data Leakage Investigation

| Requirement | How it is covered in this notebook |
|---|---|
| Target leakage and indirect leakage | Cells 11.1-11.2 |
| Features recorded after the outcome; information unavailable at prediction time | Cells 11.1, 11.3 |
| Duplicate entities across training and test | Cell 11.4 |
| Preprocessing leakage | Cell 11.5 |
| Feature-selection and hyperparameter-tuning leakage | Cell 11.6 (noise experiment) |
| Temporal leakage and leakage through aggregates | Cell 11.7 |
| Validation/test do not influence fitting | Cell 11.8 |
| Realistic splitting for the deployment scenario | Cell 11.9 |

In [54]:
lk[['column', 'verdict', 'evidence']]

,column,verdict,evidence
0,exit_date,CONFIRMED LEAKAGE,filled for 100.0% of leavers and 0.0% of stayers (filled-in flag AUC=1.0); every non-null date i...
1,resignation_date,CONFIRMED LEAKAGE,filled for 92.0% of leavers and 0.0% of stayers (filled-in flag AUC=0.9599); every non-null date...
2,exit_reason,CONFIRMED LEAKAGE,filled for 100.0% of leavers and 0.0% of stayers (filled-in flag AUC=1.0)
3,exit_interview_done,CONFIRMED LEAKAGE,filled for 100.0% of leavers and 0.0% of stayers (filled-in flag AUC=1.0)
4,fnf_settlement_amount,CONFIRMED LEAKAGE,filled for 100.0% of leavers and 0.0% of stayers (filled-in flag AUC=1.0)
5,voluntary_left_within_12m,CONFIRMED LEAKAGE,"derived from the exit event; 1 in 321 rows, all of them target=1 (rows with voluntary=1 and targ..."
6,source,SUSPECTED (excluded),"dataset-origin tag, not an employee attribute; label rates {'historical_extension': 0.15, 'maste..."
7,observation_date,SUSPECTED (excluded),time stamp of the snapshot (2021-2025 for historical rows); single-column AUC=0.504; excluded to...


In [55]:
print('indirect leakage candidates - best single-column AUC (0.5 = useless):'); single.head(6)
print('source alone:', round(A._auc_categorical(raw.source, y_all), 3), '| observation_date alone:', round(A._auc_numeric((raw.observation_date - raw.observation_date.min()).dt.days, y_all), 3), '-> excluded anyway')

indirect leakage candidates - best single-column AUC (0.5 = useless):
source alone: 0.506 | observation_date alone: 0.504 -> excluded anyway


In [56]:
pd.DataFrame([('profile, role, pay, tenure, raises, promotions', 'yes: known at the snapshot'), ('engagement / performance (last values)', 'yes: last survey/review before the snapshot'),
              ('attendance and payroll aggregates', 'yes ONLY for months ended on/before observation_date (cutoff enforced)'), ('exit_date, resignation_date, exit_reason, exit_interview_done, fnf_settlement_amount, voluntary_left', 'NO: created by the exit itself')],
             columns=['information', 'available at prediction time?'])

,information,available at prediction time?
0,"profile, role, pay, tenure, raises, promotions",yes: known at the snapshot
1,engagement / performance (last values),yes: last survey/review before the snapshot
2,attendance and payroll aggregates,yes ONLY for months ended on/before observation_date (cutoff enforced)
3,"exit_date, resignation_date, exit_reason, exit_interview_done, fnf_settlement_amount, voluntary_...",NO: created by the exit itself


In [57]:
print('train/test employee overlap:', len(ids_tr & ids_te), '| train/test row_id overlap:', len(set(dfj.loc[Xtr.index, 'row_id']) & set(dfj.loc[Xte.index, 'row_id'])))

train/test employee overlap: 0 | train/test row_id overlap: 0


In [58]:
Xa = D.clean_raw(Xtr)[ncols]; Xa = Xa.fillna(Xa.median())
wrong = Pipeline([('lr', LRB())]); Xs_all = pd.DataFrame(StandardScaler().fit_transform(Xa), index=Xa.index)       # WRONG: scaler fitted on all rows before CV
right = Pipeline([('sc', StandardScaler()), ('lr', LRB())])
print('scaling before CV (leaky):', round(roc_auc_score(ytr, cross_val_predict(wrong, Xs_all, ytr, cv=folds, method='predict_proba')[:, 1]), 4),
      '| scaling inside each fold (correct):', round(roc_auc_score(ytr, cross_val_predict(right, Xa, ytr, cv=folds, method='predict_proba')[:, 1]), 4), '(tiny gap here, huge with target-based steps - see next cell)')

scaling before CV (leaky): 0.7042 | scaling inside each fold (correct): 0.7042 (tiny gap here, huge with target-based steps - see next cell)


In [59]:
from sklearn.feature_selection import SelectKBest, f_classif
rng = np.random.default_rng(SEED); Xn = rng.normal(size=(600, 2000)); yn = rng.integers(0, 2, 600)                 # PURE NOISE: nothing is predictable
cvn = StratifiedKFold(5, shuffle=True, random_state=SEED)
leaky_sel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)                                                       # WRONG: select using ALL labels
ok_pipe = Pipeline([('sel', SelectKBest(f_classif, k=20)), ('lr', LogisticRegression(max_iter=500))])
print('feature selection BEFORE cross-validation on noise: AUC =', round(roc_auc_score(yn, cross_val_predict(LogisticRegression(max_iter=500), leaky_sel, yn, cv=cvn, method='predict_proba')[:, 1]), 3), '(looks predictive - an illusion)')
print('feature selection INSIDE cross-validation on noise : AUC =', round(roc_auc_score(yn, cross_val_predict(ok_pipe, Xn, yn, cv=cvn, method='predict_proba')[:, 1]), 3), '(correctly about 0.5)')

feature selection BEFORE cross-validation on noise: AUC = 0.733 (looks predictive - an illusion)
feature selection INSIDE cross-validation on noise : AUC = 0.481 (correctly about 0.5)


In [60]:
fut = att[att.month > pd.Timestamp('2025-09-30')]; print('attendance rows dated after the 30 Sep 2025 snapshot (future):', len(fut), '- the aggregates ignore them')
print('max month used in any aggregate:', 'Sep 2025 (cutoff: month_end <= observation_date, last 3 complete months)')
print('rows with monthly aggregates:', int(dfj.att_months_covered.notna().sum()), '| months used per row (max):', int(dfj.att_months_covered.max()), '(window = 3 complete months ending on/before observation_date)')

attendance rows dated after the 30 Sep 2025 snapshot (future): 4985 - the aggregates ignore them
max month used in any aggregate: Sep 2025 (cutoff: month_end <= observation_date, last 3 complete months)
rows with monthly aggregates: 383 | months used per row (max): 3 (window = 3 complete months ending on/before observation_date)


In [61]:
pipe_chk = F.build_pipeline(True).fit(Xtr, ytr); print('pipeline fitted on', len(Xtr), 'rows; test rows used in fit: 0; test rows scored:', len(pipe_chk.transform(Xte)))
print('test index overlaps training index:', bool(set(Xtr.index) & set(Xte.index)))

pipeline fitted on 1906 rows; test rows used in fit: 0; test rows scored: 477
test index overlaps training index: False


In [62]:
yr = dfj.observation_date.dt.year; tr_m, te_m = (yr < 2025).to_numpy(), (yr >= 2025).to_numpy()
Xall, yall = S['X'], S['y']; mdl = mk(LRB(), False).fit(Xall[tr_m], yall[tr_m]); chron = roc_auc_score(yall[te_m], mdl.predict_proba(Xall[te_m])[:, 1])
print(f'random stratified CV ROC-AUC {roc_auc_score(ytr, oof_lr):.3f}  vs  chronological (train < 2025, test 2025) ROC-AUC {chron:.3f}  (n_test={int(te_m.sum())}, leavers={int(yall[te_m].sum())})')
print('Deployment is always "train on the past, score the future", so the chronological number is the more realistic warning; random CV is kept for the main comparison because the history spans only 5 years and 2025 holds few rows.')

random stratified CV ROC-AUC 0.710  vs  chronological (train < 2025, test 2025) ROC-AUC 0.678  (n_test=485, leavers=68)
Deployment is always "train on the past, score the future", so the chronological number is the more realistic warning; random CV is kept for the main comparison because the history spans only 5 years and 2025 holds few rows.


---
---
# PHASE 4: TARGET AND FEATURE PREPARATION

---
## 12. Target Variable Preparation

| Requirement | How it is covered in this notebook |
|---|---|
| Confirm target definition and label quality; missing/invalid/inconsistent labels | Cells 12.1-12.2 |
| Positive/negative classes; binary/multiclass/... | Cell 12.1 |
| Encode labels; keep the mapping | Cell 12.1 |
| Class frequencies and imbalance | Cell 12.3 |
| Does the target definition change over time? | Cell 12.4 |
| Features must not directly reveal the target | Cell 12.5 |

In [63]:
enc = {'stayed': 0, 'left': 1}; inv_enc = {v: k for k, v in enc.items()}; print('mapping:', enc, '| unique target values:', sorted(y_all.unique().tolist()), '| dtype', y_all.dtype)

mapping: {'stayed': 0, 'left': 1} | unique target values: [0, 1] | dtype int64


In [64]:
print('missing:', int(y_all.isna().sum()), '| invalid (not 0/1):', int((~y_all.isin([0, 1])).sum()), '| conflicting per employee:', int((conf > 1).sum()))

missing: 0 | invalid (not 0/1): 0 | conflicting per employee: 0


In [65]:
print(y_all.value_counts().rename(index=inv_enc).to_dict(), '| imbalance ratio stayed:left =', round((1 - y_all.mean()) / y_all.mean(), 1), ': 1')

{'stayed': 2040, 'left': 349} | imbalance ratio stayed:left = 5.8 : 1


In [66]:
print((100 * raw.groupby(raw.observation_date.dt.year)[C.TARGET].mean()).round(1).to_dict(), '-> definition is the same 12-month rule in every year; rate is stable (no definition drift)')

{2021: 15.0, 2022: 13.5, 2023: 14.3, 2024: 16.3, 2025: 13.8} -> definition is the same 12-month rule in every year; rate is stable (no definition drift)


In [67]:
print('max single-column AUC among model inputs:', single.single_column_auc.max().round(3), '| target in X:', C.TARGET in S['X'].columns)

max single-column AUC among model inputs: 0.6 | target in X: False


---
## 13. Feature Identification

| Requirement | How it is covered in this notebook |
|---|---|
| Separate predictors from the target | Cell 13.1 |
| Numerical, categorical, datetime, text and identifier features | Cell 13.1 |
| Remove target copies and post-outcome information | Cell 13.2 |
| Relevance and reliability | Cell 13.3 |
| Redundant and irrelevant features | Cell 13.4 |
| Identifiers kept separately for traceability | Cell 13.1 |
| Features unavailable at prediction | Cell 13.2 |
| Sensitive/protected attributes | Cell 13.5 (fairness check with the official group_gap) |

In [68]:
X_all, y_enc = S['X'], S['y']; ids = dfj[['row_id', 'employee_id', 'observation_date']]            # identifiers kept OUT of X, joined back only for traceability
print('X', X_all.shape, '| y', y_enc.shape, '| ids kept separately:', list(ids.columns))
print('numeric:', len(C.NUMERIC_CORE), '| categorical:', len(C.CATEGORICAL_CORE), '| monthly aggregates:', len(C.MONTHLY_ALL), '| datetime/text/id (not model inputs): observation_date, exit_reason, row_id, employee_id')

X (2383, 43) | y (2383,) | ids kept separately: ['row_id', 'employee_id', 'observation_date']
numeric: 25 | categorical: 6 | monthly aggregates: 12 | datetime/text/id (not model inputs): observation_date, exit_reason, row_id, employee_id


In [69]:
print('removed (target copies / post-outcome / unavailable at prediction time):', C.LEAKAGE_CONFIRMED, '+ source, observation_date, ids')

removed (target copies / post-outcome / unavailable at prediction time): ['exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount', 'voluntary_left_within_12m'] + source, observation_date, ids


In [70]:
single.head(10)

,column,single_column_auc
0,months_since_last_raise,0.6000
1,last_engagement_score,0.5963
2,manager_changes_12m,0.5720
3,comp_ratio,0.5532
4,wfh_share_3m,0.5524
5,commute_distance_km,0.5495
6,raises_received,0.5410
7,tenure_months,0.5400
8,last_performance_rating,0.5392
9,work_mode,0.5359


In [71]:
cz = D.clean_raw(Xtr)[C.NUMERIC_CORE].corr().abs(); r_ = cz.where(np.triu(np.ones(cz.shape), 1).astype(bool)).stack(); print('redundant pairs |r| > 0.8:', r_[r_ > .8].round(3).to_dict() or 'none'); print('irrelevant (single-column AUC < 0.51):', list(single[single.single_column_auc < .51].column))

redundant pairs |r| > 0.8: {('tenure_months', 'raises_received'): 0.92}
irrelevant (single-column AUC < 0.51): ['employment_type', 'total_experience_years', 'internal_transfers', 'department_id', 'job_level', 'training_hours_12m', 'paid_leave_days_6m']


In [72]:
print('protected attributes (age, gender, ...) are NOT in this table and are never model inputs.')
oof_s = pd.Series(oof_lr, index=Xtr.index)
for g in ['work_mode', 'department_id']:
    means, gap = H.group_gap(oof_s.values, dfj.loc[Xtr.index, g].values); print(f'official group_gap by {g}: largest gap in mean score = {gap:.3f}', '(a fairness warning if large, not proof)')

protected attributes (age, gender, ...) are NOT in this table and are never model inputs.
official group_gap by work_mode: largest gap in mean score = 0.118 (a fairness warning if large, not proof)
official group_gap by department_id: largest gap in mean score = 0.207 (a fairness warning if large, not proof)


---
## 14. Train, Validation and Test Split

| Requirement | How it is covered in this notebook |
|---|---|
| Choose a strategy from the deployment scenario | Cell 14.1 table |
| Stratified splitting | Cell 14.2 (used) |
| Group-based splitting for repeated entities | Cell 14.3 |
| Chronological splitting | Cell 11.9 and 14.3 |
| Site/geographic splitting | Cell 14.3 |
| Train/validation/test or CV + final test | Cell 14.2: 80/20 + 5-fold CV on the 80% |
| Final test untouched during selection | Cell 14.4 (guard) |
| Class distributions and counts per partition | Cell 14.2 |
| Entity overlap and duplicate leakage | Cell 8.4 / 14.2 |
| Document strategy and seed | Cell 14.1 |

In [73]:
pd.DataFrame([('stratified random 80/20 + stratified 5-fold CV', 'MAIN: employees are unique, class is rare, history spans 5 years'), ('group split by employee', 'checked: zero employee overlap'),
              ('chronological (train < 2025)', 'sanity check of realism (Cell 11.9)'), ('site hold-out (one city)', 'generalisation check (Cell 14.3)'), ('seed', str(SEED))], columns=['strategy', 'role'])

,strategy,role
0,stratified random 80/20 + stratified 5-fold CV,"MAIN: employees are unique, class is rare, history spans 5 years"
1,group split by employee,checked: zero employee overlap
2,chronological (train < 2025),sanity check of realism (Cell 11.9)
3,site hold-out (one city),generalisation check (Cell 14.3)
4,seed,42


In [74]:
part = pd.DataFrame({'rows': [len(Xtr), len(Xte)], 'leavers': [int(ytr.sum()), int(yte.sum())], 'leaver_rate_pct': [round(100 * ytr.mean(), 2), round(100 * yte.mean(), 2)]}, index=['train (CV inside)', 'test (final only)'])
part.loc['employee overlap'] = [len(ids_tr & ids_te), None, None]; print('positives per CV validation fold:', [int(ytr.iloc[b].sum()) for _, b in folds]); part

positives per CV validation fold: [56, 55, 56, 56, 56]


,rows,leavers,leaver_rate_pct
train (CV inside),1906.0,279.0,14.64
test (final only),477.0,70.0,14.68
employee overlap,0.0,NaN,NaN


In [75]:
te_loc = dfj.loc[Xtr.index, 'work_location'].pipe(D.canonical_city); city = te_loc.value_counts().index[0]; hold = (te_loc == city).to_numpy()
m = mk(LRB()).fit(Xtr[~hold], ytr[~hold]); print(f'site hold-out ({city}, n={int(hold.sum())}): ROC-AUC on the unseen city = {roc_auc_score(ytr[hold], m.predict_proba(Xtr[hold])[:, 1]):.3f}')
gk = GroupKFold(5); gs = dfj.loc[Xtr.index, 'employee_id'].to_numpy(); g_oof = cross_val_predict(mk(LRB()), Xtr, ytr, cv=gk.split(Xtr, ytr, gs), method='predict_proba')[:, 1]; print('GroupKFold (by employee) OOF ROC-AUC:', round(roc_auc_score(ytr, g_oof), 3))

site hold-out (Bengaluru, n=710): ROC-AUC on the unseen city = 0.721
GroupKFold (by employee) OOF ROC-AUC: 0.717


In [76]:
TEST_LOCKED = True      # the test set is only touched in Section 37
print('test rows used in any fit/selection above: 0 | test set locked:', TEST_LOCKED)

test rows used in any fit/selection above: 0 | test set locked: True


---
---
# PHASE 5: DATA PREPROCESSING

---
## 15. Numerical Feature Preprocessing

| Requirement | How it is covered in this notebook |
|---|---|
| Handle missing numerical values | Cell 7.4 + pipeline median imputer (+ indicators) |
| Validate numerical data types | Cell 15.1 |
| Scaling when required; standardization, min-max, robust | Cell 15.2 |
| Log/power transforms for skewed features | Cell 15.3 |
| Invalid and infinite values | Cell 15.1 |
| Address outliers when justified | Section 9 |
| Preserve meaning and units | Cell 15.4 |

In [77]:
nm = D.clean_raw(Xtr)[C.NUMERIC_CORE]; print('all numeric dtypes:', bool(nm.dtypes.map(pd.api.types.is_numeric_dtype).all()), '| inf values:', int(np.isinf(nm).sum().sum()), '| remaining NaN (handled by the imputer):', int(nm.isna().sum().sum()))

all numeric dtypes: True | inf values: 0 | remaining NaN (handled by the imputer): 2083


In [78]:
pd.Series({'StandardScaler (logistic regression needs scaling)': quick_auc(SimpleImputer(strategy='median'), StandardScaler()), 'MinMaxScaler': quick_auc(SimpleImputer(strategy='median'), MinMaxScaler()),
           'RobustScaler': quick_auc(SimpleImputer(strategy='median'), RobustScaler())}, name='OOF ROC-AUC').round(4)

StandardScaler (logistic regression needs scaling)    0.7049
MinMaxScaler                                          0.7098
RobustScaler                                          0.7050
Name: OOF ROC-AUC, dtype: float64

In [79]:
from sklearn.preprocessing import PowerTransformer
sk = nm.skew().abs().sort_values(ascending=False); print('most skewed inputs:', sk.head(5).round(2).to_dict())
col = sk.index[0]; v = nm[col].dropna(); print(f'{col}: skew {v.skew():.2f} -> log1p {np.log1p(v.clip(lower=0)).skew():.2f} -> Yeo-Johnson {pd.Series(PowerTransformer().fit_transform(v.to_frame()).ravel()).skew():.2f}')
print('pipeline choice: log1p on pay (strongly skewed money), StandardScaler on the rest')

most skewed inputs: {'annual_fixed_pay': 5.25, 'avg_hours_worked_3m': 4.64, 'internal_transfers': 3.45, 'avg_overtime_hours_per_month_3m': 3.03, 'paid_leave_days_6m': 2.29}
annual_fixed_pay: skew 5.25 -> log1p 0.15 -> Yeo-Johnson 0.02
pipeline choice: log1p on pay (strongly skewed money), StandardScaler on the rest


In [80]:
print('units preserved: pay in INR/year, tenure in months, rates 0-1, hours/day - scaling is applied only inside the model pipeline; reports always show original units')

units preserved: pay in INR/year, tenure in months, rates 0-1, hours/day - scaling is applied only inside the model pipeline; reports always show original units


---
## 16. Categorical Feature Preprocessing

| Requirement | How it is covered in this notebook |
|---|---|
| Missing categories | Cell 16.1 |
| Standardize names | Cell 10.1 (`canonical_city`) |
| One-hot for nominal | Cell 16.2 (used) |
| Ordinal only with genuine order | Cell 16.2 (job_level is already ordinal numeric) |
| Frequency/target encoding, leakage-safe | Cell 16.3 (cross-fitted TargetEncoder) |
| Rare categories | Cell 16.4 (min_frequency) |
| Unseen categories at inference | Cell 16.4 |
| Target not leaked into encodings | Cell 16.3 |

In [81]:
print(raw[C.CATEGORICAL_CORE].isna().sum().to_dict(), '-> missing category filled with the constant "Unknown" inside the pipeline'); print(raw.highest_education.value_counts(dropna=False).head(8).to_dict())

{'department_id': 0, 'role_id': 0, 'work_location': 0, 'work_mode': 0, 'employment_type': 0, 'highest_education': 15} -> missing category filled with the constant "Unknown" inside the pipeline
{'B.Tech Information Technology': 244, 'M.Tech': 233, 'B.E. Computer Science': 229, 'MCA': 210, 'MBA': 198, 'B.Com': 178, 'BBA': 173, 'B.Sc Computer Science': 166}


In [82]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
Xcat = D.clean_raw(Xtr)[C.CATEGORICAL_CORE].fillna('Unknown'); freq = Xcat.apply(lambda s: s.map(s.value_counts(normalize=True)))
def cat_auc(enc):
    p = Pipeline([('enc', ColumnTransformer([('c', enc, C.CATEGORICAL_CORE)])), ('lr', LRB())]); return roc_auc_score(ytr, cross_val_predict(p, Xcat, ytr, cv=folds, method='predict_proba')[:, 1])
print('categorical-only models, OOF ROC-AUC (nominal categories have no natural order, so ordinal codes are NOT used):')
print(pd.Series({'one-hot (used)': cat_auc(OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=15, sparse_output=False)),
                 'cross-fitted target encoding (leakage-safe)': cat_auc(TargetEncoder(random_state=SEED))}).round(4))

categorical-only models, OOF ROC-AUC (nominal categories have no natural order, so ordinal codes are NOT used):
one-hot (used)                                 0.5629
cross-fitted target encoding (leakage-safe)    0.5499
dtype: float64


In [83]:
print('target encoding fits per-category means INSIDE each training fold with internal cross-fitting, so a row never sees its own label. A naive full-data target encoding would leak.')

target encoding fits per-category means INSIDE each training fold with internal cross-fitting, so a row never sees its own label. A naive full-data target encoding would leak.


In [84]:
ohe = OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=15, sparse_output=False).fit(Xcat[['role_id']])
print('role_id categories kept:', len(ohe.categories_[0]) - len(ohe.infrequent_categories_[0]), '| rare ones grouped:', len(ohe.infrequent_categories_[0]))
print('unseen role at inference ->', ohe.transform(pd.DataFrame({'role_id': ['R999']})).sum(), '(falls in the rare bucket, no error)')

role_id categories kept: 39 | rare ones grouped: 9
unseen role at inference -> 1.0 (falls in the rare bucket, no error)


---
## 17. Datetime and Text Preprocessing

| Requirement | How it is covered in this notebook |
|---|---|
| Parse dates consistently | Cell 17.1 |
| Extract valid date/time features; elapsed durations | Cell 17.2 |
| Historical time-window features without future information | Cell 17.3 |
| Time zones and missing dates | Cell 17.1 |
| Clean/normalize text; suitable representations (BoW, TF-IDF, embeddings) | Cell 17.4 |
| Avoid text containing the answer or the future | Cell 17.5 |

In [85]:
dts = pd.to_datetime(raw_csv.observation_date); print('CSV dates parsed:', dts.notna().all(), '| timezone-naive (no tz handling needed):', dts.dt.tz is None, '| missing dates in inputs: 0 (exit/resignation dates are blank for stayers by design and are not inputs)')

CSV dates parsed: True | timezone-naive (no tz handling needed): True | missing dates in inputs: 0 (exit/resignation dates are blank for stayers by design and are not inputs)


In [86]:
age_in_months = (raw.observation_date - raw.observation_date.min()).dt.days / 30.4
print('elapsed duration example: months since the earliest snapshot (NOT used as a feature: a time proxy)'); age_in_months.describe().round(1).to_dict()

elapsed duration example: months since the earliest snapshot (NOT used as a feature: a time proxy)


{'count': 2389.0,
 'mean': 30.0,
 'std': 18.1,
 'min': 0.0,
 '25%': 14.2,
 '50%': 29.7,
 '75%': 45.5,
 'max': 57.0}

In [87]:
print('historical windows: last-3-complete-months aggregates, cutoff = observation_date'); dfj.loc[dfj.att_months_covered.notna(), ['employee_id', 'observation_date', 'att_attendance_rate_mean_3m', 'pay_net_to_gross_3m', 'att_months_covered']].head(3)

historical windows: last-3-complete-months aggregates, cutoff = observation_date


,employee_id,observation_date,att_attendance_rate_mean_3m,pay_net_to_gross_3m,att_months_covered
0,EMP-0002,2025-09-30,0.924233,0.906986,3.0
1,EMP-0004,2025-09-30,0.970333,0.980549,3.0
2,EMP-0011,2025-09-30,0.948600,0.951256,3.0


In [88]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
jd = H.load('job_descriptions'); tcol = max([c for c in jd.columns if pd.api.types.is_string_dtype(jd[c]) or jd[c].dtype == object], key=lambda c: jd[c].astype(str).str.len().mean())
txt = jd[tcol].astype(str).str.lower().str.replace(r'[^a-z0-9 ]', ' ', regex=True)
tf = TfidfVectorizer(stop_words='english', max_features=500).fit(txt); print(f'text column {tcol!r}: {len(jd)} documents -> TF-IDF matrix', tf.transform(txt).shape, '| bag-of-words:', CountVectorizer(stop_words='english').fit(txt).transform(txt).shape)
print('embeddings would be the next step; this text (job descriptions) does not belong to the attrition model')

text column 'description_text': 10 documents -> TF-IDF matrix (10, 172) | bag-of-words: (10, 172)
embeddings would be the next step; this text (job descriptions) does not belong to the attrition model


In [89]:
print("excluded text/date fields:", ['exit_reason (written after the exit = the answer)', 'exit_date', 'resignation_date'], '| used text fields in the model: none')

excluded text/date fields: ['exit_reason (written after the exit = the answer)', 'exit_date', 'resignation_date'] | used text fields in the model: none


---
## 18. Preprocessing Pipeline

| Requirement | How it is covered in this notebook |
|---|---|
| Combine steps in a reproducible pipeline | Cell 18.1 |
| Fit on training data only; apply to validation and test | Cell 18.2 |
| Consistency between training and production | Cell 18.3 |
| Unseen categories and missing input columns | Cell 18.4 |
| Validate the transformed matrix | Cell 18.5 |
| Prevent training-serving inconsistencies | Cell 18.3 |

In [90]:
pipe = F.build_pipeline(True); print(pipe)

Pipeline(steps=[('clean',
                 RawCleaner(expected=['job_level', 'tenure_months',
                                      'total_experience_years',
                                      'previous_employers_count',
                                      'commute_distance_km', 'annual_fixed_pay',
                                      'comp_ratio', 'raises_received',
                                      'last_raise_pct',
                                      'months_since_last_raise',
                                      'promotions_received',
                                      'months_since_last_promotion',
                                      'internal_transfers',
                                      'last_performance_rating',
                                      'last_enga...
                                                   'pay_net_to_gross_3m']),
                                                 ('cat',
                                                  Pipeline(step

In [91]:
pipe.fit(Xtr, ytr); Ztr, Zva = pipe.transform(Xtr), pipe.transform(Xte.head(50)); print('fitted on', len(Xtr), 'training rows; applied to held-out rows (not used to fit):', Zva.shape)

fitted on 1906 training rows; applied to held-out rows (not used to fit): (50, 142)


In [92]:
import joblib, tempfile, os
f = os.path.join(tempfile.mkdtemp(), 'pipe.joblib'); joblib.dump(pipe, f); same = np.allclose(joblib.load(f).transform(Xte.head(50)).to_numpy(), Zva.to_numpy())
print('saved pipeline reloaded in a fresh process gives identical features (training = serving code):', same)

saved pipeline reloaded in a fresh process gives identical features (training = serving code): True


In [93]:
t = Xte.head(3).copy(); t.loc[t.index[0], 'role_id'] = 'R999'; t = t.drop(columns=['comp_ratio', 'work_mode'])
print('unseen category + two MISSING input columns -> finite output:', bool(np.isfinite(pipe.transform(t).to_numpy()).all()))

unseen category + two MISSING input columns -> finite output: True


In [94]:
print('shape', Ztr.shape, '| NaN', int(Ztr.isna().sum().sum()), '| inf', int(np.isinf(Ztr.to_numpy()).sum()), '| constant columns', int((Ztr.nunique() <= 1).sum()), '| unique names', Ztr.columns.is_unique)
print('target in features:', any('left_within' in c for c in Ztr.columns))

shape (1906, 142) | NaN 0 | inf 0 | constant columns 0 | unique names True
target in features: False


---
---
# PHASE 6: FEATURE ENGINEERING AND FEATURE SELECTION

---
## 19. Feature Engineering

| Requirement | How it is covered in this notebook |
|---|---|
| Meaningful features from existing variables; ratios and aggregates | Cell 19.1 |
| Date components and elapsed durations | Cell 17.2 / tenure_months |
| Historical behavioural features | Cell 17.3 (3-month attendance/payroll) |
| Interaction features | Cell 19.2 (validated by CV) |
| Transform skewed variables; group categories only when justified | Cell 15.3 / rare-category grouping |
| Text features | Cell 17.4 (not applicable to attrition) |
| Every feature available at prediction time | Cell 11.3 |
| Validate new features through CV | Cell 19.2 |
| Document definitions and dependencies | Cell 19.3 |

In [95]:
d = pipe.named_steps['derive'].transform(pipe.named_steps['clean'].transform(Xtr)); new = d[['pay_vs_dept_median', 'raise_stagnation_ratio', 'never_promoted', 'overtime_flag', 'raise_recency_bucket']]
print(new.describe(include='all').T[['count', 'mean', 'unique', 'top']].round(3)); print((100 * pd.crosstab(new.raise_recency_bucket, ytr.to_numpy(), normalize='index')[1]).round(1).to_dict(), '<- leaver % by raise recency')

                         count      mean unique   top
pay_vs_dept_median      1904.0  1.010516    NaN   NaN
raise_stagnation_ratio  1903.0  0.407932    NaN   NaN
never_promoted          1906.0  0.622246    NaN   NaN
overtime_flag           1906.0  0.183106    NaN   NaN
raise_recency_bucket      1906       NaN      4  0-6m
{'0-6m': 12.4, '12-18m': 12.8, '18m+': 31.7, '6-12m': 9.1} <- leaver % by raise recency


In [96]:
Xi = D.clean_raw(Xtr)[ncols].copy(); base = quick_auc(SimpleImputer(strategy='median'))
Xi['eng_x_overtime'] = Xi.last_engagement_score * Xi.avg_overtime_hours_per_month_3m; Xi['tenure_x_raise_gap'] = Xi.tenure_months * Xi.months_since_last_raise; Xc_bak = Xc; Xc = Xi
inter = quick_auc(SimpleImputer(strategy='median')) if False else roc_auc_score(ytr, cross_val_predict(Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler()), ('lr', LRB())]), Xi, ytr, cv=folds, method='predict_proba')[:, 1]); Xc = Xc_bak
print(f'OOF ROC-AUC without interactions {base:.4f}  vs with 2 interaction features {inter:.4f}  -> keep only if it helps beyond the fold spread (about 0.02)')

OOF ROC-AUC without interactions 0.7049  vs with 2 interaction features 0.7035  -> keep only if it helps beyond the fold spread (about 0.02)


In [97]:
pd.DataFrame(F.FEATURE_DOCS, columns=['feature', 'source', 'definition', 'temporal cutoff'])

,feature,source,definition,temporal cutoff
0,pay_vs_dept_median,derived,annual pay / median pay of same department+job level; medians fitted on training rows only,snapshot at observation_date
1,raise_stagnation_ratio,derived,months_since_last_raise / (tenure_months + 1),snapshot at observation_date
2,never_promoted,derived,1 if promotions_received == 0 (explains the blank months_since_last_promotion),snapshot at observation_date
3,raise_recency_bucket,derived (level-up),"months_since_last_raise in buckets 0-6, 6-12, 12-18, 18+ months, or 'no raise on record'; one-ho...",snapshot at observation_date
4,overtime_flag,derived (level-up),1 if avg_overtime_hours_per_month_3m is above the 75th percentile of the TRAINING rows,snapshot at observation_date; threshold fitted on training only
5,log_annual_pay,derived,log1p(annual_fixed_pay) after blanking pay-typos (>3x role mid-point),snapshot at observation_date
6,att_attendance_rate_mean_3m,attendance_monthly_features,mean attendance_rate over last 3 complete months,months with month_end <= observation_date
7,att_unpaid_absence_sum_3m,attendance_monthly_features,sum unpaid_absence_days over last 3 months,same
8,att_late_arrivals_sum_3m,attendance_monthly_features,sum late_arrivals over last 3 months,same
9,att_overtime_mean_3m,attendance_monthly_features,mean overtime_hours over last 3 months,same


---
## 20. Feature Selection

| Requirement | How it is covered in this notebook |
|---|---|
| Domain knowledge removes invalid predictors | Sections 11 and 13 |
| Filter methods: correlation, chi-square, mutual information | Cell 20.1 |
| Wrapper: RFE | Cell 20.2 |
| Embedded: L1, model-based | Cell 20.3 |
| Multicollinearity and redundancy | Cell 6.6 / 13.4 |
| Remove constant/unhelpful features | Cell 20.1 (VarianceThreshold) |
| Evaluate selection inside CV | Cells 20.2-20.4 |
| Compare performance, complexity, stability, interpretability | Cell 20.4 |
| Avoid selecting with test results | Cell 20.4 (test locked) |

In [98]:
from sklearn.feature_selection import mutual_info_classif, chi2, VarianceThreshold, SelectKBest, RFE, SelectFromModel
Zt = pipe.transform(Xtr); vt = VarianceThreshold(1e-8).fit(Zt); print('constant/near-constant after preprocessing:', int((~vt.get_support()).sum()), 'of', Zt.shape[1])
mi = pd.Series(mutual_info_classif(Zt, ytr, random_state=SEED), index=Zt.columns).sort_values(ascending=False)
ch = pd.Series(chi2(MinMaxScaler().fit_transform(Zt), ytr)[0], index=Zt.columns).sort_values(ascending=False)
pd.DataFrame({'mutual_info (top)': mi.head(8).index, 'chi2 (top)': ch.head(8).index})

constant/near-constant after preprocessing: 0 of 142


,mutual_info (top),chi2 (top)
0,att_attendance_rate_mean_3m,raise_recency_bucket_18m+
1,avg_hours_worked_3m,work_mode_Onsite
2,raise_stagnation_ratio,employment_type_Fixed-Term Contract
3,att_wfh_share_3m,raise_recency_bucket_6-12m
4,highest_education_MBA,raise_stagnation_ratio
5,raise_recency_bucket_6-12m,manager_changes_12m
6,never_promoted,highest_education_B.E. Electronics
7,commute_distance_km,department_id_D06


In [99]:
sel_models = {'all features (reference)': mk(LRB()),
  'filter: SelectKBest(f_classif, k=40)': Pipeline([('features', F.build_pipeline(True)), ('sel', SelectKBest(f_classif, k=40)), ('lr', LRB())]),
  'wrapper: RFE(LR, 40 features)': Pipeline([('features', F.build_pipeline(True)), ('sel', RFE(LogisticRegression(max_iter=2000), n_features_to_select=40, step=10)), ('lr', LRB())]),
  'embedded: L1 selection': Pipeline([('features', F.build_pipeline(True)), ('sel', SelectFromModel(LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1))), ('lr', LRB())])}
sel_res = {}
for n, m in sel_models.items():
    s = cross_val_predict(m, Xtr, ytr, cv=folds, method='predict_proba')[:, 1]; sel_res[n] = {'ROC-AUC': roc_auc_score(ytr, s), 'PR-AUC': average_precision_score(ytr, s)}
pd.DataFrame(sel_res).T.round(4)    # selection happens INSIDE every CV fold

,ROC-AUC,PR-AUC
all features (reference),0.7104,0.3138
"filter: SelectKBest(f_classif, k=40)",0.7180,0.3164
"wrapper: RFE(LR, 40 features)",0.7006,0.2975
embedded: L1 selection,0.7176,0.3244


In [100]:
cnt = {}
for tr, _ in folds:
    fe = F.build_pipeline(True).fit(Xtr.iloc[tr], ytr.iloc[tr]); Zf = fe.transform(Xtr.iloc[tr]); l1 = LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1).fit(Zf, ytr.iloc[tr])
    for c in Zf.columns[l1.coef_[0] != 0]: cnt[c] = cnt.get(c, 0) + 1
stab = pd.Series(cnt).sort_values(ascending=False); print('L1 keeps', int(stab.size), 'distinct features across folds; kept in all 5 folds:', int((stab == 5).sum()), '(stability)')
stab.head(10)

L1 keeps 42 distinct features across folds; kept in all 5 folds: 14 (stability)


commute_distance_km                        5
comp_ratio                                 5
manager_changes_12m                        5
months_since_last_raise                    5
last_engagement_score                      5
last_performance_rating                    5
missingindicator_total_experience_years    5
wfh_share_3m                               5
avg_hours_worked_3m                        5
avg_overtime_hours_per_month_3m            5
dtype: int64

In [101]:
print('complexity vs performance: the 40-feature subsets score within the fold-to-fold spread (about 0.02 ROC-AUC) of the full ~150-feature model, so regularised logistic regression needs no hard selection; interpretability is highest with the small L1 subset.')
print('test set locked: no selection used it.')

complexity vs performance: the 40-feature subsets score within the fold-to-fold spread (about 0.02 ROC-AUC) of the full ~150-feature model, so regularised logistic regression needs no hard selection; interpretability is highest with the small L1 subset.
test set locked: no selection used it.


---
## 21. Dimensionality Reduction

| Requirement | How it is covered in this notebook |
|---|---|
| Is the feature count excessive? | Cell 21.1 |
| PCA for suitable numeric data | Cell 21.2 |
| Fit reduction on training only | Cell 21.2 (inside the pipeline) |
| Interpretability lost? | Cell 21.3 |
| Compare before/after | Cell 21.3 |

In [102]:
from sklearn.decomposition import PCA
print('model-ready features:', Zt.shape[1], 'for', len(Xtr), 'rows (ratio', round(len(Xtr) / Zt.shape[1], 1), 'rows per feature)')
p = PCA().fit(Zt); cum = np.cumsum(p.explained_variance_ratio_); print('components for 80% / 90% / 95% variance:', [int(np.argmax(cum >= q) + 1) for q in (.8, .9, .95)])

model-ready features: 142 for 1906 rows (ratio 13.4 rows per feature)
components for 80% / 90% / 95% variance: [28, 37, 50]


In [103]:
res = {}
for k in (10, 20, 40):
    m = Pipeline([('features', F.build_pipeline(True)), ('pca', PCA(n_components=k, random_state=SEED)), ('lr', LRB())])   # PCA fitted inside each training fold
    res[f'PCA {k} components'] = roc_auc_score(ytr, cross_val_predict(m, Xtr, ytr, cv=folds, method='predict_proba')[:, 1])
res['no reduction (all features)'] = roc_auc_score(ytr, oof_lr); pd.Series(res, name='OOF ROC-AUC').round(4)

PCA 10 components              0.6209
PCA 20 components              0.6681
PCA 40 components              0.6988
no reduction (all features)    0.7104
Name: OOF ROC-AUC, dtype: float64

In [104]:
print('PCA components mix many original columns, so coefficients no longer say "raise age" or "pay gap": interpretability drops with no AUC gain -> PCA is NOT used in the final pipeline.')

PCA components mix many original columns, so coefficients no longer say "raise age" or "pay gap": interpretability drops with no AUC gain -> PCA is NOT used in the final pipeline.


---
---
# PHASE 7: CLASS IMBALANCE HANDLING

---
## 22. Class Imbalance Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Class counts and proportions | Cell 22.1 |
| Does imbalance affect the business objective? | Cell 22.2 |
| Majority-class baseline performance | Cell 22.2 |
| Cost of missing minority examples | Cell 1.3 / 22.2 |
| Imbalance across time or subgroups | Cell 22.3 |

In [105]:
print(y_all.value_counts().to_dict(), '| proportions', y_all.value_counts(normalize=True).round(4).to_dict())

{0: 2040, 1: 349} | proportions {0: 0.8539, 1: 0.1461}


In [106]:
maj = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr); pr = maj.predict(Xte if False else Xtr)
print(f'majority baseline: accuracy {accuracy_score(ytr, pr):.3f}, recall {recall_score(ytr, pr):.0f} -> misses all {int(ytr.sum())} leavers; at {cost_fn} cost units per missed leaver that is {ytr.sum() * cost_fn:.0f} units of avoidable loss on the training rows')

majority baseline: accuracy 0.854, recall 0 -> misses all 279 leavers; at 5.0 cost units per missed leaver that is 1395 units of avoidable loss on the training rows


In [107]:
sub = pd.concat([dfj.groupby(dfj.observation_date.dt.year)[C.TARGET].mean().round(3).rename('by year'), ], axis=1); print(sub.T.to_string())
print(dfj.groupby('department_id')[C.TARGET].mean().round(3).agg(['min', 'max']).to_dict(), '<- leaver rate range across departments (imbalance differs by subgroup)')

observation_date  2021   2022   2023   2024  2025
by year           0.15  0.135  0.143  0.163  0.14
{'min': 0.065, 'max': 0.211} <- leaver rate range across departments (imbalance differs by subgroup)


---
## 23. Imbalance Handling Techniques

| Requirement | How it is covered in this notebook |
|---|---|
| Class weights | Cell 23.1 |
| Random oversampling / undersampling | Cell 23.1 |
| SMOTE or variants | Cell 23.1 |
| Threshold adjustment | Cell 23.2 |
| Collect more representative minority examples | Cell 23.3 (recommendation) |
| Effects of resampling on calibration and generalisation | Cell 23.1 (Brier, mean probability) |
| Resample only the training part of each fold; never before splitting | Cell 23.1 code |
| Compare against an unmodified baseline | Cell 23.1 |

In [108]:
rng = np.random.default_rng(SEED)
def cv_resample(kind):
    oof = np.zeros(len(ytr))
    for tr, te in folds:
        fe = F.build_pipeline(True).fit(Xtr.iloc[tr], ytr.iloc[tr]); Z1, Z2 = fe.transform(Xtr.iloc[tr]).to_numpy(), fe.transform(Xtr.iloc[te]).to_numpy(); y1 = ytr.iloc[tr].to_numpy()   # resampling touches TRAINING rows of the fold only
        pos, neg = np.where(y1 == 1)[0], np.where(y1 == 0)[0]
        if kind == 'random oversampling': idx = np.r_[neg, rng.choice(pos, len(neg))]; Z1, y1 = Z1[idx], y1[idx]
        elif kind == 'random undersampling': idx = np.r_[pos, rng.choice(neg, len(pos), replace=False)]; Z1, y1 = Z1[idx], y1[idx]
        elif kind == 'SMOTE': Z1, y1 = M.SimpleSMOTE(random_state=SEED).fit_resample(Z1, y1)
        lr = LogisticRegression(max_iter=2000, class_weight='balanced' if kind == 'class weights' else None, random_state=SEED).fit(Z1, y1); oof[te] = lr.predict_proba(Z2)[:, 1]
    return oof
imb = {k: cv_resample(k) for k in ['unmodified (no weights)', 'class weights', 'random oversampling', 'random undersampling', 'SMOTE']}
pd.DataFrame({k: {'ROC-AUC': roc_auc_score(ytr, s), 'PR-AUC': average_precision_score(ytr, s), 'recall@0.5': recall_score(ytr, s >= .5), 'precision@0.5': precision_score(ytr, s >= .5, zero_division=0),
                  'Brier': brier_score_loss(ytr, s), 'mean predicted prob': s.mean()} for k, s in imb.items()}).T.round(3)

,ROC-AUC,PR-AUC,recall@0.5,precision@0.5,Brier,mean predicted prob
unmodified (no weights),0.713,0.324,0.161,0.450,0.118,0.148
class weights,0.710,0.314,0.584,0.260,0.200,0.380
random oversampling,0.706,0.307,0.556,0.261,0.199,0.367
random undersampling,0.685,0.269,0.656,0.236,0.241,0.434
SMOTE,0.698,0.302,0.556,0.261,0.201,0.360


Reading: ranking quality is similar for every strategy (ROC-AUC 0.69-0.71; random undersampling is a little worse because it throws data away); resampling and weights mainly **shift the probabilities upward** (mean predicted probability far above the real 14.6%, worse Brier), so they change threshold behaviour and calibration, not the ordering. The unmodified model is best calibrated.

In [109]:
s0 = imb['unmodified (no weights)']; ths = np.arange(.05, .4, .025); f1s = [f1_score(ytr, s0 >= t) for t in ths]; best_t = ths[int(np.argmax(f1s))]
print(f'threshold adjustment instead of resampling: unmodified LR, best-F1 threshold {best_t:.3f} -> recall {recall_score(ytr, s0 >= best_t):.2f}, precision {precision_score(ytr, s0 >= best_t):.2f}')

threshold adjustment instead of resampling: unmodified LR, best-F1 threshold 0.250 -> recall 0.44, precision 0.33


**Collect more representative minority examples:** the most valuable fix is more real leavers (and monthly history for the 2,000 historical rows), not more synthetic ones. SMOTE here interpolates across one-hot columns, which is a known limitation (SMOTE-NC would be better).

---
---
# PHASE 8: BASELINE AND MODEL SELECTION

---
## 24. Establish a Baseline

| Requirement | How it is covered in this notebook |
|---|---|
| Majority-class or prior-probability baseline | Cell 24.1 |
| Baseline metric suited to the class distribution | Cell 24.1 |
| Record baseline performance | Cell 24.1 |
| Minimum acceptable performance | Cell 24.2 |
| Compare ML models against the baseline | Cell 24.3 / Section 26 |

In [110]:
base = {}
for nm, strat in [('majority class', 'most_frequent'), ('prior probability', 'prior')]:
    s = cross_val_predict(mk(DummyClassifier(strategy=strat)), Xtr, ytr, cv=folds, method='predict_proba')[:, 1]
    base[nm] = {'ROC-AUC': roc_auc_score(ytr, s), 'PR-AUC': average_precision_score(ytr, s), 'recall@0.5': recall_score(ytr, s >= .5), 'accuracy': accuracy_score(ytr, s >= .5)}
print('baseline metric: PR-AUC (a useless model scores the base rate, 0.146) and ROC-AUC (0.5), NOT accuracy'); pd.DataFrame(base).T.round(3)

baseline metric: PR-AUC (a useless model scores the base rate, 0.146) and ROC-AUC (0.5), NOT accuracy


,ROC-AUC,PR-AUC,recall@0.5,accuracy
majority class,0.500,0.146,0.0,0.854
prior probability,0.498,0.146,0.0,0.854


In [111]:
MIN_OK = {'ROC-AUC': 0.70, 'top-10% lift': 2.0, 'PR-AUC above base rate by': 0.10}
print('minimum acceptable (from the business brief + supplied benchmark 0.713):', MIN_OK)

minimum acceptable (from the business brief + supplied benchmark 0.713): {'ROC-AUC': 0.7, 'top-10% lift': 2.0, 'PR-AUC above base rate by': 0.1}


In [112]:
o = H.binary_report(ytr, oof_lr); print({k: round(v, 3) for k, v in o.items()}, f'\nlift in top 10%: {o["precision_at_top"] / o["base_rate"]:.2f}x vs baseline 1.0x -> LR beats the baselines')

{'roc_auc': 0.71, 'pr_auc': 0.314, 'precision_at_top': 0.374, 'recall_at_top': 0.254, 'f1_at_top': 0.303, 'base_rate': np.float64(0.146)} 
lift in top 10%: 2.55x vs baseline 1.0x -> LR beats the baselines


---
## 25. Select Candidate Classification Algorithms

| Requirement | How it is covered in this notebook |
|---|---|
| Logistic Regression | Cell 25.1 |
| KNN | Cell 25.1 |
| Decision Tree | Cell 25.1 |
| Random Forest | Cell 25.1 |
| Extra Trees | Cell 25.1 |
| SVM | Cell 25.1 |
| Naive Bayes | Cell 25.1 |
| Gradient Boosting | Cell 25.1 |
| AdaBoost | Cell 25.1 |
| XGBoost / LightGBM / CatBoost | Cell 25.1 (guarded import: runs where installed; not installed in the build sandbox) |
| Neural Networks | Cell 25.1 (MLP) |
| CNN / RNN / transformer | Cell 25.2: not suitable for a tabular snapshot (explained) |
| Cost, latency, explainability, missing-value support, deployment | Cell 25.3 table |

In [113]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
zoo = {'Dummy (prior)': DummyClassifier(strategy='prior'), 'Logistic Regression (balanced)': LRB(), 'KNN (k=25)': KNeighborsClassifier(25),
       'Decision Tree (depth 4, balanced)': DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=SEED),
       'Random Forest (300, balanced)': RandomForestClassifier(300, class_weight='balanced', min_samples_leaf=3, random_state=SEED),
       'Extra Trees (300, balanced)': ExtraTreesClassifier(300, class_weight='balanced', min_samples_leaf=3, random_state=SEED),
       'SVM (RBF, balanced)': SVC(probability=True, class_weight='balanced', random_state=SEED), 'Naive Bayes (Gaussian)': GaussianNB(),
       'Gradient Boosting (150)': GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=.05, random_state=SEED), 'AdaBoost (150)': AdaBoostClassifier(n_estimators=150, random_state=SEED),
       'HistGradientBoosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=150, random_state=SEED),
       'Neural network (MLP 32)': MLPClassifier((32,), alpha=1e-2, early_stopping=True, max_iter=500, random_state=SEED)}
skipped = {}
for nm, mod, kw in [('XGBoost', 'xgboost', 'XGBClassifier'), ('LightGBM', 'lightgbm', 'LGBMClassifier'), ('CatBoost', 'catboost', 'CatBoostClassifier')]:
    try:
        m_ = __import__(mod); params = dict(n_estimators=200, max_depth=3, learning_rate=.05, random_state=SEED); zoo[nm] = getattr(m_, kw)(**({**params, 'verbose': 0} if nm == 'CatBoost' else params if nm != 'LightGBM' else {**params, 'verbose': -1}))
    except Exception as e: skipped[nm] = f'not installed here ({type(e).__name__}); code above runs where `pip install {mod}` is available'
print(len(zoo), 'candidate models:', list(zoo)); print('skipped:', skipped or 'none')

12 candidate models: ['Dummy (prior)', 'Logistic Regression (balanced)', 'KNN (k=25)', 'Decision Tree (depth 4, balanced)', 'Random Forest (300, balanced)', 'Extra Trees (300, balanced)', 'SVM (RBF, balanced)', 'Naive Bayes (Gaussian)', 'Gradient Boosting (150)', 'AdaBoost (150)', 'HistGradientBoosting', 'Neural network (MLP 32)']
skipped: {'XGBoost': 'not installed here (ModuleNotFoundError); code above runs where `pip install xgboost` is available', 'LightGBM': 'not installed here (ModuleNotFoundError); code above runs where `pip install lightgbm` is available', 'CatBoost': 'not installed here (ModuleNotFoundError); code above runs where `pip install catboost` is available'}


**25.2 Deep-learning families.** CNNs target images, RNNs/transformers target sequences or text. The attrition input is one tabular snapshot per employee (about 40 mixed numeric/categorical columns, 2,383 rows), so those architectures add cost without a plausible benefit; a small MLP is included as the neural-network representative.

In [114]:
pd.DataFrame([('Logistic Regression', 'low', '<1 ms', 'high', 'needs imputation', 'trivial'), ('Decision Tree / Random Forest / Extra Trees', 'low-medium', '1-20 ms', 'medium-high (feature importance)', 'needs imputation here', 'easy'),
              ('Gradient Boosting / HistGB / XGBoost / LightGBM / CatBoost', 'medium', '1-10 ms', 'medium (needs SHAP)', 'HistGB/XGB/LGBM/CatBoost handle NaN natively', 'easy'), ('KNN / SVM', 'medium', 'slow at scale', 'low', 'needs imputation', 'heavier'),
              ('Naive Bayes', 'very low', '<1 ms', 'medium', 'needs imputation', 'trivial'), ('Neural network (MLP)', 'medium', '1 ms', 'low', 'needs imputation', 'moderate'), ('CNN / RNN / transformer', 'high', 'higher', 'low', 'n/a', 'not suitable for this tabular task')],
             columns=['algorithm family', 'compute cost', 'inference latency', 'explainability', 'missing-value support', 'deployment effort'])

,algorithm family,compute cost,inference latency,explainability,missing-value support,deployment effort
0,Logistic Regression,low,<1 ms,high,needs imputation,trivial
1,Decision Tree / Random Forest / Extra Trees,low-medium,1-20 ms,medium-high (feature importance),needs imputation here,easy
2,Gradient Boosting / HistGB / XGBoost / LightGBM / CatBoost,medium,1-10 ms,medium (needs SHAP),HistGB/XGB/LGBM/CatBoost handle NaN natively,easy
3,KNN / SVM,medium,slow at scale,low,needs imputation,heavier
4,Naive Bayes,very low,<1 ms,medium,needs imputation,trivial
5,Neural network (MLP),medium,1 ms,low,needs imputation,moderate
6,CNN / RNN / transformer,high,higher,low,n/a,not suitable for this tabular task


---
---
# PHASE 9: MODEL TRAINING

---
## 26. Initial Model Training

| Requirement | How it is covered in this notebook |
|---|---|
| Fit each candidate on the training data with the pipeline | Cell 26.1 |
| Consistent evaluation strategy | Cell 26.1 (same 5 folds for every model) |
| Record training time and compute | Cell 26.1 table |
| Track seeds and settings | Cell 26.1 (seed 42, settings printed) |
| Initial validation performance; compare on the primary metric | Cell 26.2 |
| Convergence warnings and fitting failures | Cell 26.1 (warnings captured, not hidden) |

In [115]:
oof_all, rows, wlog = {}, [], {}
for nm, est in zoo.items():
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always'); t0 = time.time()
        try: s = cross_val_predict(mk(est), Xtr, ytr, cv=folds, method='predict_proba')[:, 1]
        except Exception as e: rows.append({'model': nm, 'status': f'FAILED: {type(e).__name__}'}); continue
        sec = time.time() - t0
    wlog[nm] = sorted({f'{x.category.__name__}: {str(x.message)[:70]}' for x in w}); oof_all[nm] = s; o = H.binary_report(ytr, s)
    fold_auc = [roc_auc_score(ytr.iloc[b], s[b]) for _, b in folds]
    rows.append({'model': nm, 'status': 'ok', 'ROC-AUC': np.mean(fold_auc), 'ROC sd': np.std(fold_auc, ddof=1), 'PR-AUC': o['pr_auc'], 'top10 precision': o['precision_at_top'], 'top10 recall': o['recall_at_top'], 'recall@0.5': recall_score(ytr, s >= .5), 'seconds (5 folds)': sec})
init = pd.DataFrame(rows); print('seed', SEED, '| identical folds | warnings captured:', {k: v for k, v in wlog.items() if v} or 'none'); init.sort_values('PR-AUC', ascending=False).round(3)

seed 42 | identical folds | warnings captured: none


,model,status,ROC-AUC,ROC sd,PR-AUC,top10 precision,top10 recall,recall@0.5,seconds (5 folds)
9,AdaBoost (150),ok,0.715,0.016,0.321,0.353,0.240,0.079,10.142
1,Logistic Regression (balanced),ok,0.709,0.021,0.314,0.374,0.254,0.584,2.061
6,"SVM (RBF, balanced)",ok,0.713,0.017,0.298,0.342,0.233,0.043,11.011
10,HistGradientBoosting,ok,0.695,0.015,0.292,0.368,0.251,0.050,4.415
8,Gradient Boosting (150),ok,0.708,0.037,0.292,0.347,0.237,0.043,12.316
4,"Random Forest (300, balanced)",ok,0.700,0.025,0.287,0.368,0.251,0.093,13.242
5,"Extra Trees (300, balanced)",ok,0.684,0.020,0.268,0.326,0.222,0.222,11.740
3,"Decision Tree (depth 4, balanced)",ok,0.629,0.037,0.230,0.337,0.229,0.534,1.374
2,KNN (k=25),ok,0.639,0.025,0.223,0.274,0.186,0.000,1.497
11,Neural network (MLP 32),ok,0.596,0.072,0.197,0.200,0.136,0.014,2.860


In [116]:
print('failures:', init[init.status != 'ok'].to_dict('records') or 'none')
print('best by primary metric (PR-AUC):', init.loc[init['PR-AUC'].idxmax(), 'model'], '| best ROC-AUC:', init.loc[init['ROC-AUC'].idxmax(), 'model'])

failures: none
best by primary metric (PR-AUC): AdaBoost (150) | best ROC-AUC: AdaBoost (150)


---
## 27. Cross-Validation

| Requirement | How it is covered in this notebook |
|---|---|
| Choose an appropriate CV method | Cell 27.1 |
| Stratified k-fold | Cell 27.1 (main) |
| Group CV for grouped entities | Cell 14.3 |
| Time-series / forward-chaining for temporal problems | Cell 27.2 |
| Repeated CV | Cell 27.3 |
| Mean and variability of scores | Cells 27.1, 27.3 |
| Folds contain enough positives | Cell 27.1 |
| Preprocessing/selection inside the CV pipeline | all models are Pipeline objects |
| Final test not used | Cell 14.4 |

In [117]:
fa = [roc_auc_score(ytr.iloc[b], oof_lr[b]) for _, b in folds]; print(f'stratified 5-fold ROC-AUC per fold {np.round(fa, 3).tolist()} -> mean {np.mean(fa):.3f} +/- {np.std(fa, ddof=1):.3f}; leavers per validation fold {[int(ytr.iloc[b].sum()) for _, b in folds]}')

stratified 5-fold ROC-AUC per fold [0.702, 0.704, 0.734, 0.681, 0.726] -> mean 0.709 +/- 0.021; leavers per validation fold [56, 55, 56, 56, 56]


In [118]:
from sklearn.model_selection import TimeSeriesSplit
order = np.argsort(dfj.loc[Xtr.index, 'observation_date'].to_numpy(), kind='stable'); Xo, yo = Xtr.iloc[order], ytr.iloc[order]; scs = []
for a, b in TimeSeriesSplit(5).split(Xo): scs.append(roc_auc_score(yo.iloc[b], mk(LRB(), False).fit(Xo.iloc[a], yo.iloc[a]).predict_proba(Xo.iloc[b])[:, 1]))     # early periods have no monthly history -> core features
print('forward-chaining (past -> next period) ROC-AUC per split:', np.round(scs, 3).tolist(), '| mean', round(np.mean(scs), 3))

forward-chaining (past -> next period) ROC-AUC per split: [0.635, 0.73, 0.704, 0.702, 0.709] | mean 0.696


In [119]:
rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
cv_r = cross_validate(mk(LRB()), Xtr, ytr, cv=rep, scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}); print(f"repeated 5x3 CV: ROC-AUC {cv_r['test_roc_auc'].mean():.3f} +/- {cv_r['test_roc_auc'].std():.3f} | PR-AUC {cv_r['test_pr_auc'].mean():.3f} +/- {cv_r['test_pr_auc'].std():.3f} (15 fits)")

repeated 5x3 CV: ROC-AUC 0.717 +/- 0.039 | PR-AUC 0.335 +/- 0.061 (15 fits)


---
---
# PHASE 10: HYPERPARAMETER OPTIMIZATION

---
## 28. Hyperparameter Tuning

| Requirement | How it is covered in this notebook |
|---|---|
| Most influential hyperparameters; valid ranges | Cell 28.1 |
| Grid Search | Cell 28.2 |
| Randomized Search | Cell 28.3 |
| Bayesian optimization | Cell 28.4 (guarded: needs optuna/scikit-optimize; successive halving shown instead) |
| Tune with cross-validation on a metric aligned to the business | Cells 28.2-28.3 (scoring = PR-AUC) |
| Compare default vs tuned | Cell 28.5 |
| Track computational cost | Cell 28.5 |
| Avoid over-tuning a small validation set | Cell 28.6 (nested CV) |
| Select by validation evidence, never test | Cell 28.5 |

**28.1 Influential hyperparameters:** logistic regression `C` (regularisation strength) and class weight; random forest `max_depth`, `min_samples_leaf`, `max_features`, `n_estimators`; gradient boosting `learning_rate`, `max_depth`, `n_estimators`. Ranges below are the search spaces.

In [120]:
t0 = time.time(); gs = GridSearchCV(mk(LRB()), {'model__C': [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]}, scoring='average_precision', cv=folds, n_jobs=1).fit(Xtr, ytr); t_grid = time.time() - t0
print('GridSearchCV best:', gs.best_params_, f'CV PR-AUC {gs.best_score_:.4f}', f'({t_grid:.0f}s)'); pd.DataFrame(gs.cv_results_)[['param_model__C', 'mean_test_score', 'std_test_score']].round(4)

GridSearchCV best: {'model__C': 0.03} CV PR-AUC 0.3434 (13s)


,param_model__C,mean_test_score,std_test_score
0,0.003,0.3306,0.0436
1,0.010,0.3396,0.0443
2,0.030,0.3434,0.0438
3,0.100,0.3382,0.0370
4,0.300,0.3334,0.0403
5,1.000,0.3245,0.0457
6,3.000,0.3214,0.0481
7,10.000,0.3171,0.0498


In [121]:
from scipy.stats import randint, uniform
t0 = time.time(); rs = RandomizedSearchCV(mk(RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=1)), {'model__n_estimators': randint(100, 400), 'model__max_depth': randint(3, 14), 'model__min_samples_leaf': randint(1, 15), 'model__max_features': uniform(.1, .6)},
                                      n_iter=10, scoring='average_precision', cv=folds, random_state=SEED, n_jobs=1).fit(Xtr, ytr); t_rand = time.time() - t0
print('RandomizedSearchCV (10 draws) best:', {k.replace('model__', ''): (round(v, 3) if isinstance(v, float) else v) for k, v in rs.best_params_.items()}, f'CV PR-AUC {rs.best_score_:.4f} ({t_rand:.0f}s)')

RandomizedSearchCV (10 draws) best: {'max_depth': 11, 'max_features': np.float64(0.275), 'min_samples_leaf': 11, 'n_estimators': 269} CV PR-AUC 0.3218 (220s)


In [122]:
try:
    import optuna; print('optuna available: Bayesian optimisation could be run here')
except Exception:
    print('Bayesian optimisation (optuna / scikit-optimize) is NOT installed in this environment; successive halving, the budget-aware alternative, is shown instead:')
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
t0 = time.time(); hs = HalvingRandomSearchCV(mk(GradientBoostingClassifier(random_state=SEED)), {'model__learning_rate': [.02, .05, .1], 'model__max_depth': [1, 2, 3], 'model__n_estimators': [50, 100, 200], 'model__subsample': [.7, 1.0]},
                                              scoring='average_precision', cv=3, random_state=SEED, n_candidates=12, factor=3, min_resources=600, resource='n_samples', n_jobs=1).fit(Xtr, ytr); t_halv = time.time() - t0
print('HalvingRandomSearchCV best:', hs.best_params_, f'CV PR-AUC {hs.best_score_:.4f} ({t_halv:.0f}s)')

Bayesian optimisation (optuna / scikit-optimize) is NOT installed in this environment; successive halving, the budget-aware alternative, is shown instead:
HalvingRandomSearchCV best: {'model__subsample': 0.7, 'model__n_estimators': 100, 'model__max_depth': 2, 'model__learning_rate': 0.1} CV PR-AUC 0.3160 (50s)


In [123]:
def cvs(m): s = cross_val_predict(m, Xtr, ytr, cv=folds, method='predict_proba')[:, 1]; return {'ROC-AUC': roc_auc_score(ytr, s), 'PR-AUC': average_precision_score(ytr, s)}
cmp28 = pd.DataFrame({'LR default (C=1)': cvs(mk(LRB())), f'LR tuned ({gs.best_params_})': cvs(gs.best_estimator_), 'RF default': cvs(mk(RandomForestClassifier(300, class_weight='balanced', min_samples_leaf=3, random_state=SEED))),
                      'RF tuned (random search)': cvs(rs.best_estimator_), 'GB default': cvs(mk(GradientBoostingClassifier(random_state=SEED))), 'GB tuned (halving)': cvs(hs.best_estimator_)}).T
cmp28['search seconds'] = [None, t_grid, None, t_rand, None, t_halv]; cmp28.round(4)

,ROC-AUC,PR-AUC,search seconds
LR default (C=1),0.7104,0.3138,NaN
LR tuned ({'model__C': 0.03}),0.7141,0.3314,12.7778
RF default,0.6989,0.2869,NaN
RF tuned (random search),0.6998,0.2924,219.5465
GB default,0.6953,0.2862,NaN
GB tuned (halving),0.7100,0.3059,49.5997


In [124]:
outer = StratifiedKFold(5, shuffle=True, random_state=SEED + 1); nested = []
for a, b in outer.split(Xtr, ytr):
    g = GridSearchCV(mk(LRB()), {'model__C': [0.01, 0.1, 1, 10]}, scoring='average_precision', cv=3).fit(Xtr.iloc[a], ytr.iloc[a]); nested.append(average_precision_score(ytr.iloc[b], g.predict_proba(Xtr.iloc[b])[:, 1]))
print(f'nested CV (tuning inside, scoring outside) PR-AUC: {np.mean(nested):.3f} +/- {np.std(nested, ddof=1):.3f} -> an honest estimate of a TUNED model; gains from tuning are inside the noise')
best_C = gs.best_params_['model__C']; print('selected from CV only (test untouched): logistic regression C =', best_C)

nested CV (tuning inside, scoring outside) PR-AUC: 0.341 +/- 0.040 -> an honest estimate of a TUNED model; gains from tuning are inside the noise
selected from CV only (test untouched): logistic regression C = 0.03


---
## 29. Threshold Optimization

| Requirement | How it is covered in this notebook |
|---|---|
| Thresholds change predicted labels | Cell 29.1 |
| Evaluate by business cost | Cell 29.1 |
| Optimise precision, recall, F-beta or expected cost | Cell 29.1 |
| Operational capacity constraints | Cell 29.2 |
| Select the threshold on training-derived out-of-fold predictions | Cell 29.1 |
| Never choose it using test labels | Cell 29.3 |
| Reassess when prevalence/conditions change | Cell 29.4 |

In [125]:
s = oof_lr; ths = np.round(np.arange(.2, .9, .02), 2); tab = []
for t in ths:
    p = s >= t; tn, fp, fn, tp = confusion_matrix(ytr, p, labels=[0, 1]).ravel()
    tab.append({'threshold': t, 'flagged_%': 100 * p.mean(), 'precision': precision_score(ytr, p, zero_division=0), 'recall': recall_score(ytr, p), 'F1': f1_score(ytr, p), 'F2': fbeta_score(ytr, p, beta=2), 'expected_cost': fn * cost_fn + fp * cost_fp})
tt = pd.DataFrame(tab); best_cost, best_f1, best_f2 = (float(tt.loc[tt[c].idxmin() if c == 'expected_cost' else tt[c].idxmax(), 'threshold']) for c in ('expected_cost', 'F1', 'F2'))
print(f'thresholds chosen on TRAINING out-of-fold scores: min expected cost {best_cost} | max F1 {best_f1} | max F2 (recall-weighted) {best_f2}'); tt.iloc[::4].round(3)

thresholds chosen on TRAINING out-of-fold scores: min expected cost 0.24 | max F1 0.66 | max F2 (recall-weighted) 0.28


,threshold,flagged_%,precision,recall,F1,F2,expected_cost
0,0.20,66.684,0.189,0.860,0.310,0.503,710.5
4,0.28,56.086,0.210,0.806,0.334,0.515,692.0
8,0.36,46.170,0.228,0.720,0.347,0.504,729.5
12,0.44,38.353,0.244,0.638,0.352,0.482,781.5
16,0.52,31.165,0.266,0.566,0.362,0.462,823.0
20,0.60,24.187,0.286,0.473,0.357,0.419,899.5
24,0.68,17.419,0.334,0.398,0.363,0.383,950.5
28,0.76,11.490,0.365,0.287,0.321,0.300,1064.5
32,0.84,7.083,0.407,0.197,0.266,0.220,1160.0


In [126]:
K = int(.10 * len(ytr)); cap_thr = float(np.sort(s)[::-1][K - 1]); pc = s >= cap_thr
print(f'capacity rule: HR can review {K} people per cycle -> threshold {cap_thr:.3f}: precision {precision_score(ytr, pc):.3f}, recall {recall_score(ytr, pc):.3f}')
THRESHOLD = best_cost; print('operating threshold used for the confusion matrix and final test:', THRESHOLD, '(cost-based)')

capacity rule: HR can review 190 people per cycle -> threshold 0.791: precision 0.374, recall 0.254
operating threshold used for the confusion matrix and final test: 0.24 (cost-based)


In [127]:
print('test labels used to pick any threshold: NO. Thresholds above come from out-of-fold predictions of the training split only.')

test labels used to pick any threshold: NO. Thresholds above come from out-of-fold predictions of the training split only.


In [128]:
print('reassessment trigger: population stability index of scores between cycles:', round(H.psi(oof_lr, np.random.default_rng(0).permutation(oof_lr)[:500]), 4), '(<0.10 stable); re-tune the threshold if prevalence (14.6%) or PSI > 0.25 changes')

reassessment trigger: population stability index of scores between cycles: 0.0127 (<0.10 stable); re-tune the threshold if prevalence (14.6%) or PSI > 0.25 changes


---
---
# PHASE 11: MODEL EVALUATION

---
## 30. Confusion Matrix

| Requirement | How it is covered in this notebook |
|---|---|
| TP, TN, FP, FN definitions | Cell 30.1 (annotated) |
| Types and costs of errors | Cell 30.2 |

In [129]:
pred = oof_lr >= THRESHOLD; tn, fp, fn, tp = confusion_matrix(ytr, pred, labels=[0, 1]).ravel()
print(f'TP (leavers correctly flagged) = {tp} | TN (stayers correctly not flagged) = {tn} | FP (stayers wrongly flagged) = {fp} | FN (leavers missed) = {fn}')
W.plot_confusion(ytr, oof_lr, THRESHOLD, 'Out-of-fold, cost-based threshold');

TP (leavers correctly flagged) = 235 | TN (stayers correctly not flagged) = 693 | FP (stayers wrongly flagged) = 934 | FN (leavers missed) = 44


In [130]:
pd.DataFrame({'count': [tp, tn, fp, fn], 'unit cost': [0, 0, cost_fp, cost_fn]}, index=['TP', 'TN', 'FP', 'FN']).assign(total_cost=lambda d: d['count'] * d['unit cost'])

,count,unit cost,total_cost
TP,235,0.0,0.0
TN,693,0.0,0.0
FP,934,0.5,467.0
FN,44,5.0,220.0


---
## 31. Classification Metrics

| Requirement | How it is covered in this notebook |
|---|---|
| Accuracy, precision, recall/sensitivity, specificity | Cell 31.1 |
| F1, F-beta, balanced accuracy | Cell 31.1 |
| ROC-AUC, PR-AUC/average precision | Cell 31.1 |
| Log loss, Brier score | Cell 31.1 |
| MCC, Cohen's kappa | Cell 31.1 |
| Top-k accuracy, Hamming loss, subset accuracy (multiclass/multilabel) | Cell 31.2 (not applicable to a binary single-label task; computed for completeness) |
| Calibration measures | Cell 31.3 (ECE) and Section 33 |

In [131]:
s = oof_lr; p = s >= THRESHOLD
M31 = {'accuracy': (accuracy_score(ytr, p), 'overall share correct (misleading at 14.6% base rate)'), 'precision': (precision_score(ytr, p), 'flagged who truly left'), 'recall / sensitivity': (recall_score(ytr, p), 'leavers who were flagged'),
       'specificity': (tn / (tn + fp), 'stayers correctly not flagged'), 'F1': (f1_score(ytr, p), 'harmonic mean of precision and recall'), 'F2 (recall-weighted)': (fbeta_score(ytr, p, beta=2), 'recall counts 4x more than precision'),
       'balanced accuracy': (balanced_accuracy_score(ytr, p), 'mean of recall and specificity'), 'ROC-AUC': (roc_auc_score(ytr, s), 'chance a leaver outranks a stayer'), 'PR-AUC / average precision': (average_precision_score(ytr, s), 'list purity; base rate is 0.146'),
       'log loss': (log_loss(ytr, np.clip(s, 1e-6, 1 - 1e-6)), 'penalises confident wrong probabilities'), 'Brier score': (brier_score_loss(ytr, s), 'squared probability error'),
       'MCC': (matthews_corrcoef(ytr, p), 'balanced correlation from all four counts'), "Cohen's kappa": (cohen_kappa_score(ytr, p), 'agreement beyond chance')}
pd.DataFrame({k: {'value': v[0], 'meaning': v[1]} for k, v in M31.items()}).T

,value,meaning
accuracy,0.486884,overall share correct (misleading at 14.6% base rate)
precision,0.201027,flagged who truly left
recall / sensitivity,0.842294,leavers who were flagged
specificity,0.425937,stayers correctly not flagged
F1,0.324586,harmonic mean of precision and recall
F2 (recall-weighted),0.514223,recall counts 4x more than precision
balanced accuracy,0.634116,mean of recall and specificity
ROC-AUC,0.710398,chance a leaver outranks a stayer
PR-AUC / average precision,0.313784,list purity; base rate is 0.146
log loss,0.597335,penalises confident wrong probabilities


In [132]:
topk_prec = H.binary_report(ytr, s)['precision_at_top']
print(f'top-k accuracy: meaningful for multiclass only; for this binary task the equivalent is precision in the top 10% = {topk_prec:.3f}')
print(f'Hamming loss = {hamming_loss(ytr, p):.3f} (= 1 - accuracy for one binary label) | subset accuracy = {accuracy_score(ytr, p):.3f} (exact label match = accuracy here) -> multilabel metrics reduce to accuracy and are not decision-relevant')

top-k accuracy: meaningful for multiclass only; for this binary task the equivalent is precision in the top 10% = 0.374
Hamming loss = 0.513 (= 1 - accuracy for one binary label) | subset accuracy = 0.487 (exact label match = accuracy here) -> multilabel metrics reduce to accuracy and are not decision-relevant


In [133]:
def ece(y, s, bins=10):
    q = pd.qcut(s, bins, duplicates='drop'); g = pd.DataFrame({'y': np.asarray(y), 's': s}).groupby(q, observed=True).agg(n=('y', 'size'), y=('y', 'mean'), s=('s', 'mean')); return float((g.n * (g.y - g.s).abs()).sum() / g.n.sum())
print(f'expected calibration error (raw scores): {ece(ytr, s):.3f}  -> large because class_weight=balanced inflates probabilities (see Section 33)')

expected calibration error (raw scores): 0.236  -> large because class_weight=balanced inflates probabilities (see Section 33)


---
## 32. Metric Selection

| Requirement | How it is covered in this notebook |
|---|---|
| Accuracy when meaningful | Cell 32.1 |
| Precision when false positives are costly; recall when misses are costly; F1/F-beta | Cell 32.1 |
| PR-AUC for rare positives; ROC-AUC for ranking | Cell 32.1 |
| Log loss, Brier, calibration plots when probabilities matter | Cell 32.1 |
| Per-class precision/recall/F1; macro, micro, weighted averages | Cell 32.2 |
| Business cost, capacity, subgroup performance | Cell 32.3 |
| Do not choose with accuracy alone | Cell 32.1 |

In [134]:
pd.DataFrame([('accuracy', 'only if classes are balanced and errors cost the same', 'NOT used (85% for a model that finds nobody)'), ('precision', 'false alarms are expensive (manager time)', 'reported'), ('recall', 'missing a leaver is expensive', 'reported (cost ratio 10:1)'),
              ('F1 / F-beta', 'one number balancing both', 'F2 used for threshold search'), ('PR-AUC', 'rare positives', 'PRIMARY'), ('ROC-AUC', 'ranking across thresholds', 'secondary / benchmark comparison'),
              ('log loss, Brier, calibration', 'probabilities are shown or used as numbers', 'reported; bands used instead of raw probabilities'), ('top-10% recall / lift', 'review capacity is limited', 'PRIMARY (business list)')],
             columns=['metric', 'use when', 'decision here'])

,metric,use when,decision here
0,accuracy,only if classes are balanced and errors cost the same,NOT used (85% for a model that finds nobody)
1,precision,false alarms are expensive (manager time),reported
2,recall,missing a leaver is expensive,reported (cost ratio 10:1)
3,F1 / F-beta,one number balancing both,F2 used for threshold search
4,PR-AUC,rare positives,PRIMARY
5,ROC-AUC,ranking across thresholds,secondary / benchmark comparison
6,"log loss, Brier, calibration",probabilities are shown or used as numbers,reported; bands used instead of raw probabilities
7,top-10% recall / lift,review capacity is limited,PRIMARY (business list)


In [135]:
from sklearn.metrics import classification_report
rep = pd.DataFrame(classification_report(ytr, p, target_names=['stayed', 'left'], output_dict=True)).T.round(3); rep    # per-class + macro / weighted averages (micro = accuracy for single-label binary)

,precision,recall,f1-score,support
stayed,0.940,0.426,0.586,1627.000
left,0.201,0.842,0.325,279.000
accuracy,0.487,0.487,0.487,0.487
macro avg,0.571,0.634,0.455,1906.000
weighted avg,0.832,0.487,0.548,1906.000


In [136]:
sub = dfj.loc[Xtr.index].assign(pred=p, y=ytr.to_numpy())
pd.concat([sub.groupby(g).apply(lambda d: pd.Series({'n': len(d), 'recall': recall_score(d.y, d.pred, zero_division=0), 'precision': precision_score(d.y, d.pred, zero_division=0)}), include_groups=False).assign(group=g) for g in ['work_mode', 'department_id']]).round(3)

,n,recall,precision,group
Hybrid,904.0,0.802,0.166,work_mode
Onsite,646.0,0.891,0.241,work_mode
Remote,356.0,0.800,0.197,work_mode
D01,650.0,0.822,0.178,department_id
D02,140.0,0.800,0.211,department_id
D03,198.0,0.926,0.227,department_id
D04,209.0,0.714,0.168,department_id
D05,80.0,0.778,0.149,department_id
D06,235.0,0.896,0.257,department_id
D07,71.0,0.750,0.188,department_id


---
## 33. Probability and Calibration Evaluation

| Requirement | How it is covered in this notebook |
|---|---|
| Predicted probability distribution | Cell 33.1 |
| Reliability diagram | Cell 33.2 |
| Brier score / log loss | Cell 33.2 |
| Platt scaling or isotonic regression | Cell 33.3 |
| Calibration fitted via suitable CV; evaluated on data not used to fit it | Cell 33.3 (nested) |
| Do probabilities support the business decision? | Cell 33.4 |

In [137]:
fig, ax = plt.subplots(figsize=(7, 3.6)); ax.hist(oof_lr, bins=40); ax.axvline(ytr.mean(), c='r', ls='--', label='true base rate'); ax.legend(); ax.set_xlabel('predicted probability (raw)'); ax.set_title('Predicted probability distribution')

Text(0.5, 1.0, 'Predicted probability distribution')

In [138]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
cal = {'raw': oof_lr}
for meth in ('sigmoid', 'isotonic'):    # calibration is fitted INSIDE each outer training fold (cv=3 inside) and scored on the outer validation rows
    cal[meth] = cross_val_predict(CalibratedClassifierCV(mk(LRB()), method=meth, cv=3), Xtr, ytr, cv=folds, method='predict_proba')[:, 1]
fig, ax = plt.subplots(figsize=(5.5, 5)); ax.plot([0, 1], [0, 1], 'k--')
for k, v in cal.items(): fp_, mp_ = calibration_curve(ytr, v, n_bins=10, strategy='quantile'); ax.plot(mp_, fp_, 'o-', label=k)
ax.legend(); ax.set_xlabel('mean predicted'); ax.set_ylabel('observed rate'); ax.set_title('Reliability diagram (out-of-fold)')
pd.DataFrame({k: {'Brier': brier_score_loss(ytr, v), 'log loss': log_loss(ytr, np.clip(v, 1e-6, 1 - 1e-6)), 'ECE': ece(ytr, v), 'mean prob': v.mean(), 'ROC-AUC': roc_auc_score(ytr, v)} for k, v in cal.items()}).T.round(4)

,Brier,log loss,ECE,mean prob,ROC-AUC
raw,0.2000,0.5973,0.2355,0.3796,0.7104
sigmoid,0.1155,0.3830,0.0230,0.1487,0.7089
isotonic,0.1153,0.3823,0.0198,0.1498,0.7093


In [139]:
print('calibrated (sigmoid/isotonic) probabilities average about', round(cal['sigmoid'].mean(), 3), 'vs true', round(ytr.mean(), 3), '; ranking (ROC-AUC) is unchanged by Platt scaling.')
print('mapping fitted on inner training folds only; evaluated on outer validation rows never used to fit it.')

calibrated (sigmoid/isotonic) probabilities average about 0.149 vs true 0.146 ; ranking (ROC-AUC) is unchanged by Platt scaling.
mapping fitted on inner training folds only; evaluated on outer validation rows never used to fit it.


In [140]:
print('Decision support: managers should use RISK BANDS (rank-based), not raw probabilities; if a probability is ever shown, show the calibrated one (about 15% average), because a raw balanced-model "0.40" overstates risk.')

Decision support: managers should use RISK BANDS (rank-based), not raw probabilities; if a probability is ever shown, show the calibrated one (about 15% average), because a raw balanced-model "0.40" overstates risk.


---
## 34. Error Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Inspect false positives and false negatives | Cell 34.1 |
| Recurring error patterns | Cell 34.2 |
| Difficult examples, ambiguous labels | Cell 34.3 |
| Errors by group, geography, time, segment | Cell 34.4 |
| Poor data quality and missing features | Cell 34.5 |
| Distribution shift between training and evaluation data | Cell 34.6 |
| Model limits vs label noise vs collection process | Cell 34.7 |
| Prioritise fixes by impact | Cell 34.8 |

In [141]:
err = dfj.loc[Xtr.index].assign(score=oof_lr, pred=p, y=ytr.to_numpy()); err['kind'] = np.select([(err.pred) & (err.y == 1), (err.pred) & (err.y == 0), (~err.pred) & (err.y == 1)], ['TP', 'FP', 'FN'], 'TN')
cols = ['row_id', 'department_id', 'tenure_months', 'comp_ratio', 'months_since_last_raise', 'last_engagement_score', 'score']
print('highest-scored FALSE POSITIVES (flagged but stayed)'); print(err[err.kind == 'FP'].nlargest(6, 'score')[cols].round(2).to_string(index=False))
print('\nlowest-scored FALSE NEGATIVES (leavers the model was most sure would stay)'); err[err.kind == 'FN'].nsmallest(6, 'score')[cols].round(2)

highest-scored FALSE POSITIVES (flagged but stayed)
 row_id department_id  tenure_months  comp_ratio  months_since_last_raise  last_engagement_score  score
   1564           D06             24        1.30                       24                    2.6   0.99
   1544           D06              3        0.91                        3                    NaN   0.98
    577           D08              9        1.09                        9                    2.8   0.97
   1211           D02              6        0.99                        6                    1.7   0.97
    302           D06             13        0.91                        5                    2.3   0.97
   1056           D04             18        1.12                       18                    2.1   0.96

lowest-scored FALSE NEGATIVES (leavers the model was most sure would stay)


,row_id,department_id,tenure_months,comp_ratio,months_since_last_raise,last_engagement_score,score
168,169,D01,34,1.04,5,3.1,0.00
327,328,D04,8,0.99,8,3.1,0.00
761,762,D05,58,1.10,14,3.1,0.01
1723,1724,D06,22,0.97,7,4.6,0.02
531,532,D04,21,0.93,3,4.3,0.02
1742,1743,D01,121,1.00,8,3.2,0.02


In [142]:
fe = ['tenure_months', 'comp_ratio', 'months_since_last_raise', 'last_engagement_score', 'avg_overtime_hours_per_month_3m', 'annual_fixed_pay']
err.groupby('kind')[fe].mean().round(2)    # FN look like TN (quiet leavers); FP look like TP (risky-looking stayers)

,tenure_months,comp_ratio,months_since_last_raise,last_engagement_score,avg_overtime_hours_per_month_3m,annual_fixed_pay
kind,,,,,,
FN,36.82,1.05,8.16,3.51,1.08,1192701.00
FP,33.26,1.03,11.47,3.26,2.09,1055067.67
TN,45.16,1.06,8.99,3.78,1.79,1057429.73
TP,31.79,1.01,15.23,3.08,3.62,952014.64


In [143]:
print('ambiguous labels / difficult examples: leavers who had already resigned before the snapshot (the outcome was decided when features were measured):')
rb = err.resignation_date.notna() & (err.resignation_date <= err.observation_date); print(pd.crosstab(err.kind, rb).rename(columns={False: 'not yet resigned', True: 'already resigned'}).loc[['TP', 'FN']])

ambiguous labels / difficult examples: leavers who had already resigned before the snapshot (the outcome was decided when features were measured):
col_0  not yet resigned  already resigned
kind                                     
TP                  206                29
FN                   36                 8


In [144]:
seg = err.assign(tenure_band=pd.cut(err.tenure_months.where(err.tenure_months >= 0), [0, 12, 24, 48, 400], labels=['<1y', '1-2y', '2-4y', '4y+']), city=D.canonical_city(err.work_location))
res = []
for g in ['department_id', 'tenure_band', 'city', 'source']:
    t = seg.groupby(g, observed=True).apply(lambda d: pd.Series({'n': len(d), 'FN rate (of leavers)': (d.kind == 'FN').sum() / max((d.y == 1).sum(), 1), 'FP rate (of stayers)': (d.kind == 'FP').sum() / max((d.y == 0).sum(), 1)}), include_groups=False); res.append(t.assign(segment=g).reset_index().rename(columns={g: 'group'}))
pd.concat(res).round(3).head(24)

,group,n,FN rate (of leavers),FP rate (of stayers),segment
0,D01,650.0,0.178,0.609,department_id
1,D02,140.0,0.200,0.500,department_id
2,D03,198.0,0.074,0.497,department_id
3,D04,209.0,0.286,0.547,department_id
4,D05,80.0,0.222,0.563,department_id
5,D06,235.0,0.104,0.663,department_id
6,D07,71.0,0.250,0.413,department_id
7,D08,89.0,0.200,0.581,department_id
8,D09,181.0,0.032,0.640,department_id
9,D10,53.0,0.333,0.400,department_id


In [145]:
print('missingness among errors (share of rows with a blank):'); err.assign(missing_engagement=err.last_engagement_score.isna(), missing_raise=err.months_since_last_raise.isna(), no_monthly_history=err.att_months_covered.isna()).groupby('kind')[['missing_engagement', 'missing_raise', 'no_monthly_history']].mean().round(3)

missingness among errors (share of rows with a blank):


,missing_engagement,missing_raise,no_monthly_history
kind,,,
FN,0.091,0.0,0.705
FP,0.254,0.0,0.849
TN,0.165,0.0,0.808
TP,0.281,0.0,0.881


In [146]:
older, newer = (err.observation_date.dt.year < 2025).to_numpy(), (err.observation_date.dt.year >= 2025).to_numpy()
psi_t = pd.Series({c: H.psi(err.loc[older, c].fillna(err.loc[older, c].median()), err.loc[newer, c].fillna(err.loc[older, c].median())) for c in fe}).sort_values(ascending=False)
print('population stability (PSI) of key inputs: training rows before 2025 vs 2025 rows (<0.10 stable, 0.10-0.25 watch, >0.25 drift):'); psi_t.round(3)

population stability (PSI) of key inputs: training rows before 2025 vs 2025 rows (<0.10 stable, 0.10-0.25 watch, >0.25 drift):


months_since_last_raise            1.663
tenure_months                      0.190
annual_fixed_pay                   0.174
comp_ratio                         0.074
last_engagement_score              0.064
avg_overtime_hours_per_month_3m    0.020
dtype: float64

**34.7 Where do the errors come from?** (1) *Model limits*: the signals are weak (AUC about 0.7), so many leavers look like stayers; (2) *label noise*: 12% of leavers had already resigned before the snapshot, which no safe feature can see; (3) *data collection*: 84% of rows have no attendance/payroll history, and blanks in engagement/raise data hide signal.

In [147]:
pd.DataFrame([('1', 'obtain monthly attendance/payroll history for the 2,000 historical rows', 'high: 84% of rows are blind to the strongest behavioural signals'), ('2', 'add a leave/resignation-notice field available BEFORE the snapshot', 'high: explains the 42 already-resigned leavers'),
              ('3', 'add manager-level and survey-recency features', 'medium'), ('4', 'tune the threshold per business cost', 'medium (cheap)'), ('5', 'gradient boosting / more tuning', 'low: ceiling set by data, not algorithm')], columns=['priority', 'fix', 'expected impact'])

,priority,fix,expected impact
0,1,"obtain monthly attendance/payroll history for the 2,000 historical rows",high: 84% of rows are blind to the strongest behavioural signals
1,2,add a leave/resignation-notice field available BEFORE the snapshot,high: explains the 42 already-resigned leavers
2,3,add manager-level and survey-recency features,medium
3,4,tune the threshold per business cost,medium (cheap)
4,5,gradient boosting / more tuning,"low: ceiling set by data, not algorithm"


---
## 35. Overfitting and Underfitting Analysis

| Requirement | How it is covered in this notebook |
|---|---|
| Compare training, validation and cross-validation performance | Cell 35.1 |
| Large train-validation gap | Cell 35.1 |
| Overly complex models; regularisation, pruning, simpler models | Cells 35.2-35.3 |
| Poor performance on both training and validation | Cell 35.1 (Dummy) |
| Improve features/capacity/data when justified | Cell 35.4 |
| Leakage giving unrealistically high performance | Cell 35.5 |
| Prefer robust generalisation | Cell 35.4 |

In [148]:
fit_ms = {'Dummy': zoo['Dummy (prior)'], 'Logistic Regression': zoo['Logistic Regression (balanced)'], 'Decision Tree depth 4': zoo['Decision Tree (depth 4, balanced)'],
          'Decision Tree unlimited': DecisionTreeClassifier(random_state=SEED), 'Random Forest': zoo['Random Forest (300, balanced)'], 'Gradient Boosting': zoo['Gradient Boosting (150)']}
rows = []
for n, est in fit_ms.items():
    m = mk(est).fit(Xtr, ytr); tr_auc = roc_auc_score(ytr, m.predict_proba(Xtr)[:, 1]); cv_auc = roc_auc_score(ytr, cross_val_predict(mk(est), Xtr, ytr, cv=folds, method='predict_proba')[:, 1])
    rows.append({'model': n, 'train ROC-AUC (in-sample)': tr_auc, 'CV ROC-AUC': cv_auc, 'gap': tr_auc - cv_auc, 'diagnosis': 'OVERFIT (memorises the training rows)' if tr_auc - cv_auc > .25 else 'mild overfit (gap 0.10-0.25)' if tr_auc - cv_auc > .10 else 'no signal (underfit baseline)' if cv_auc < .55 else 'well fitted'})
fit_tab = pd.DataFrame(rows); fit_tab.round(3)

,model,train ROC-AUC (in-sample),CV ROC-AUC,gap,diagnosis
0,Dummy,0.500,0.498,0.002,no signal (underfit baseline)
1,Logistic Regression,0.840,0.710,0.129,mild overfit (gap 0.10-0.25)
2,Decision Tree depth 4,0.759,0.621,0.139,mild overfit (gap 0.10-0.25)
3,Decision Tree unlimited,1.000,0.542,0.458,OVERFIT (memorises the training rows)
4,Random Forest,1.000,0.699,0.301,OVERFIT (memorises the training rows)
5,Gradient Boosting,0.856,0.704,0.152,mild overfit (gap 0.10-0.25)


In [149]:
ts, trs, vas = learning_curve(mk(LRB()), Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='roc_auc', train_sizes=np.linspace(.2, 1, 5))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); ax[0].plot(ts, trs.mean(1), 'o-', label='train'); ax[0].plot(ts, vas.mean(1), 'o-', label='validation'); ax[0].legend(); ax[0].set_title('Learning curve (logistic regression)'); ax[0].set_xlabel('training rows')
dp = [2, 3, 4, 6, 8, 12]; trc, vac = validation_curve(mk(DecisionTreeClassifier(class_weight='balanced', random_state=SEED)), Xtr, ytr, param_name='model__max_depth', param_range=dp, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='roc_auc')
ax[1].plot(dp, trc.mean(1), 'o-', label='train'); ax[1].plot(dp, vac.mean(1), 'o-', label='validation'); ax[1].legend(); ax[1].set_title('Validation curve: tree depth (overfit as depth grows)'); ax[1].set_xlabel('max_depth'); fig.tight_layout()

In [150]:
print('regularisation: logistic regression CV PR-AUC by C ->', {c: round(s, 4) for c, s in zip(gs.cv_results_['param_model__C'].data, gs.cv_results_['mean_test_score'])}); print('pruning: shallow trees (depth 3-4) beat deep trees on validation; simpler models generalise at least as well')

regularisation: logistic regression CV PR-AUC by C -> {np.float64(0.003): np.float64(0.3306), np.float64(0.01): np.float64(0.3396), np.float64(0.03): np.float64(0.3434), np.float64(0.1): np.float64(0.3382), np.float64(0.3): np.float64(0.3334), np.float64(1.0): np.float64(0.3245), np.float64(3.0): np.float64(0.3214), np.float64(10.0): np.float64(0.3171)}
pruning: shallow trees (depth 3-4) beat deep trees on validation; simpler models generalise at least as well


In [151]:
lr_row = fit_tab[fit_tab.model == 'Logistic Regression'].iloc[0]; m03 = mk(LRB(C=best_C)).fit(Xtr, ytr); a03 = roc_auc_score(ytr, m03.predict_proba(Xtr)[:, 1]); c03 = roc_auc_score(ytr, cross_val_predict(mk(LRB(C=best_C)), Xtr, ytr, cv=folds, method='predict_proba')[:, 1])
print(f"logistic regression C=1: train {lr_row['train ROC-AUC (in-sample)']:.3f} vs CV {lr_row['CV ROC-AUC']:.3f} (mild overfit from ~150 features); stronger regularisation C={best_C}: train {a03:.3f} vs CV {c03:.3f} -> smaller gap, same CV score")
print('underfitting check: more capacity does not help (forests/boosting do not beat the simple model), so the limit is the data and its signal: effort goes into features/data, not bigger models')

logistic regression C=1: train 0.840 vs CV 0.710 (mild overfit from ~150 features); stronger regularisation C=0.03: train 0.807 vs CV 0.714 -> smaller gap, same CV score
underfitting check: more capacity does not help (forests/boosting do not beat the simple model), so the limit is the data and its signal: effort goes into features/data, not bigger models


In [152]:
print('leakage check: highest honest CV ROC-AUC =', round(fit_tab['CV ROC-AUC'].max(), 3), '< 0.90 ->', 'no suspicious score; the deliberate leaky demo scored 1.00 in Task 3')

leakage check: highest honest CV ROC-AUC = 0.71 < 0.90 -> no suspicious score; the deliberate leaky demo scored 1.00 in Task 3


---
## 36. Model Comparison and Selection

| Requirement | How it is covered in this notebook |
|---|---|
| Compare all candidates on consistent splits and metrics | Cell 36.1 |
| CV mean and variability | Cell 36.1 |
| Minority-class performance | Cell 36.1 (recall/precision) |
| Calibration, interpretability, robustness | Cell 36.1 |
| Inference speed, memory, maintenance | Cell 36.1 (seconds, table in 25.3) |
| Fairness and subgroup performance | Cell 36.2 (official group_gap) |
| Cost of false positives and false negatives | Cell 36.1 (expected cost) |
| Select the model that best meets the full objective | Cell 36.3 |

In [153]:
interp = {'Dummy (prior)': 0, 'Logistic Regression (balanced)': 3, 'Decision Tree (depth 4, balanced)': 3, 'Naive Bayes (Gaussian)': 2, 'Random Forest (300, balanced)': 2, 'Extra Trees (300, balanced)': 2, 'Gradient Boosting (150)': 1, 'HistGradientBoosting': 1, 'AdaBoost (150)': 1,
          'KNN (k=25)': 1, 'SVM (RBF, balanced)': 0, 'Neural network (MLP 32)': 0}
def min_cost(s):
    return min(((s >= t) & (ytr == 0)).sum() * cost_fp + ((s < t) & (ytr == 1)).sum() * cost_fn for t in np.arange(.02, .98, .02))
comp = init[init.status == 'ok'].set_index('model')[['ROC-AUC', 'ROC sd', 'PR-AUC', 'top10 recall', 'recall@0.5', 'seconds (5 folds)']].copy()
comp['Brier'] = [brier_score_loss(ytr, oof_all[m]) for m in comp.index]; comp['min expected cost'] = [min_cost(oof_all[m]) for m in comp.index]; comp['interpretability (0-3)'] = [interp.get(m, 1) for m in comp.index]
comp.sort_values('PR-AUC', ascending=False).round(3)

,ROC-AUC,ROC sd,PR-AUC,top10 recall,recall@0.5,seconds (5 folds),Brier,min expected cost,interpretability (0-3)
model,,,,,,,,,
AdaBoost (150),0.715,0.016,0.321,0.240,0.079,10.142,0.193,697.0,1
Logistic Regression (balanced),0.709,0.021,0.314,0.254,0.584,2.061,0.200,687.0,3
"SVM (RBF, balanced)",0.713,0.017,0.298,0.233,0.043,11.011,0.116,662.0,0
HistGradientBoosting,0.695,0.015,0.292,0.251,0.050,4.415,0.117,711.0,1
Gradient Boosting (150),0.708,0.037,0.292,0.237,0.043,12.316,0.117,698.5,1
"Random Forest (300, balanced)",0.700,0.025,0.287,0.251,0.093,13.242,0.123,702.5,2
"Extra Trees (300, balanced)",0.684,0.020,0.268,0.222,0.222,11.740,0.142,751.5,2
"Decision Tree (depth 4, balanced)",0.629,0.037,0.230,0.229,0.534,1.374,0.233,805.5,3
KNN (k=25),0.639,0.025,0.223,0.186,0.000,1.497,0.122,782.0,1


In [154]:
gg = {m: H.group_gap(oof_all[m], dfj.loc[Xtr.index, 'work_mode'].values)[1] for m in ['Logistic Regression (balanced)', 'Random Forest (300, balanced)', 'Gradient Boosting (150)']}
print('official group_gap (largest gap in mean score between work modes; a fairness WARNING, not proof):', {k: round(v, 3) for k, v in gg.items()})

official group_gap (largest gap in mean score between work modes; a fairness WARNING, not proof): {'Logistic Regression (balanced)': 0.118, 'Random Forest (300, balanced)': 0.054, 'Gradient Boosting (150)': 0.048}


In [155]:
best = comp['PR-AUC'].max(); close = comp[comp['PR-AUC'] >= best - 0.01]; final_name = close.sort_values(['interpretability (0-3)', 'PR-AUC'], ascending=False).index[0]
print(f'rule: among models within 0.01 PR-AUC of the best ({best:.3f}), choose the most interpretable -> candidates: {list(close.index)}')
print('SELECTED:', final_name, '| why: ranking quality equal to complex models, transparent coefficients, calibrates easily, fast, no extra dependencies')

rule: among models within 0.01 PR-AUC of the best (0.321), choose the most interpretable -> candidates: ['Logistic Regression (balanced)', 'AdaBoost (150)']
SELECTED: Logistic Regression (balanced) | why: ranking quality equal to complex models, transparent coefficients, calibrates easily, fast, no extra dependencies


---
---
# PHASE 12: FINAL TESTING AND VALIDATION

---
## 37. Final Model Training

| Requirement | How it is covered in this notebook |
|---|---|
| Freeze the selected model design and hyperparameters | Cell 37.1 (then the held-out test is scored once) |
| (the supplied workflow text ends here: no further sub-bullets were provided) | Cells 37.2-37.5 add the standard final steps: single test evaluation, artifact + metadata saved, success-criteria check, conclusions |

In [156]:
FINAL_EST = LRB(C=best_C) if final_name.startswith('Logistic') else zoo[final_name]
FROZEN = {'model': final_name, 'estimator': repr(FINAL_EST), 'features': 'F.build_pipeline(include_monthly=True)', 'seed': SEED, 'split': 'stratified 80/20 + stratified 5-fold on the 80%', 'threshold_rule': 'minimum expected cost on training out-of-fold scores'}
print(FROZEN)
oof_final = cross_val_predict(mk(FINAL_EST), Xtr, ytr, cv=folds, method='predict_proba')[:, 1]
tt2 = pd.DataFrame([{'t': t, 'cost': ((oof_final >= t) & (ytr == 0)).sum() * cost_fp + ((oof_final < t) & (ytr == 1)).sum() * cost_fn} for t in np.arange(.2, .9, .02)]); THR_FINAL = float(tt2.loc[tt2.cost.idxmin(), 't'])
print('threshold fixed BEFORE touching the test set:', round(THR_FINAL, 2))

{'model': 'Logistic Regression (balanced)', 'estimator': "LogisticRegression(C=0.03, class_weight='balanced', max_iter=2000,\n                   random_state=42)", 'features': 'F.build_pipeline(include_monthly=True)', 'seed': 42, 'split': 'stratified 80/20 + stratified 5-fold on the 80%', 'threshold_rule': 'minimum expected cost on training out-of-fold scores'}
threshold fixed BEFORE touching the test set: 0.34


In [157]:
final = mk(FINAL_EST).fit(Xtr, ytr); p_test = final.predict_proba(Xte)[:, 1]        # the ONLY time the test labels are used
off = H.binary_report(yte, p_test); pt = p_test >= THR_FINAL; tn_, fp_, fn_, tp_ = confusion_matrix(yte, pt, labels=[0, 1]).ravel()
final_report = {**{k: round(v, 4) for k, v in off.items()}, 'threshold': round(THR_FINAL, 2), 'precision@thr': round(precision_score(yte, pt, zero_division=0), 4), 'recall@thr': round(recall_score(yte, pt), 4), 'F1@thr': round(f1_score(yte, pt), 4),
                'accuracy@thr': round(accuracy_score(yte, pt), 4), 'Brier': round(brier_score_loss(yte, p_test), 4), 'TP': int(tp_), 'FP': int(fp_), 'FN': int(fn_), 'TN': int(tn_)}
pd.Series(final_report, name='held-out test (single evaluation)').to_frame()

,held-out test (single evaluation)
roc_auc,0.6965
pr_auc,0.3714
precision_at_top,0.4043
recall_at_top,0.2714
f1_at_top,0.3248
base_rate,0.1468
threshold,0.3400
precision@thr,0.1937
recall@thr,0.7857
F1@thr,0.3107


In [158]:
bands = pd.Series(H.risk_bands(p_test)); bt = E.band_summary(yte, bands); print('official risk bands on the test set:'); print(bt.round(2).to_string(index=False))
W.plot_confusion(yte, p_test, THR_FINAL, 'Held-out test, frozen model');

official risk bands on the test set:
  band  employees  leavers  share_of_employees_pct  leaver_rate_pct  share_of_all_leavers_pct  lift_vs_base_rate
  High         48       19                   10.06            39.58                     27.14               2.70
Medium         95       18                   19.92            18.95                     25.71               1.29
   Low        334       33                   70.02             9.88                     47.14               0.67


In [159]:
lift = off['precision_at_top'] / off['base_rate']
crit = {'ROC-AUC >= 0.70': off['roc_auc'] >= 0.70, 'top-10% lift >= 2': lift >= 2.0, 'PR-AUC at least 0.10 above base rate': off['pr_auc'] - off['base_rate'] >= 0.10, 'no score above 0.90 (no leakage)': off['roc_auc'] < 0.90}
print(pd.Series({k: ('MET' if v else 'NOT MET') for k, v in crit.items()}).to_string()); print(f"(test ROC-AUC {off['roc_auc']:.3f}, lift {lift:.2f}x, PR-AUC {off['pr_auc']:.3f} vs base rate {off['base_rate']:.3f}; only {int(yte.sum())} test leavers, so differences of a few points are noise)")

ROC-AUC >= 0.70                         NOT MET
top-10% lift >= 2                           MET
PR-AUC at least 0.10 above base rate        MET
no score above 0.90 (no leakage)            MET
(test ROC-AUC 0.697, lift 2.75x, PR-AUC 0.371 vs base rate 0.147; only 70 test leavers, so differences of a few points are noise)


In [160]:
import json, joblib
joblib.dump(final, ROOT / 'artifacts' / 'models' / 'final_attrition_model.joblib')
meta = {**FROZEN, 'train_rows': len(Xtr), 'test_rows': len(Xte), 'test_report': final_report, 'success_criteria': {k: bool(v) for k, v in crit.items()}, 'versions': R.package_versions(), 'intended_use': 'decision support for HR review only; never an automatic employment decision'}
(ROOT / 'artifacts' / 'metadata' / 'final_model_card_data.json').write_text(json.dumps(meta, indent=2, default=str)); print('saved artifact and metadata')

saved artifact and metadata


## Final conclusions and recommendations
**Conclusions**
- The problem is feasible but hard: attrition signals are weak (ROC-AUC about 0.7). A balanced logistic regression is as good as every complex model tried, is explainable, and fast.
- The audit mattered: six post-outcome columns would have produced a perfect-looking (AUC 1.0) but useless model; salary typos, duplicates and impossible values were cleaned; no honest model passed 0.90.
- Class weights, SMOTE and resampling change probabilities and thresholds, not ranking; calibration is needed before any probability is shown.
- The top-10% list is about 2.5-2.75x better than random (out-of-fold and held-out test), which is the practical value for HR; the single test ROC-AUC (about 0.70, 70 leavers) sits right at the 0.70 target, so treat it as 'about 0.7', not as a precise pass/fail.

**Recommendations**
1. Use the output as a ranked, human-reviewed list (risk bands), never as a decision; keep protected attributes out and monitor group gaps with `group_gap`.
2. Collect monthly attendance/payroll history for all rows and a resignation-notice flag known before the snapshot (biggest data gaps).
3. Re-check score stability monthly with `psi`, and re-tune the threshold when prevalence or capacity changes.
4. Next experiments: gradient boosting with native missing handling (install xgboost/lightgbm), SHAP explanations, Bayesian tuning (install optuna) - expected gains are small versus the data fixes in (2).
5. Keep the test set locked: it was used exactly once, in Section 37.

*Note: the workflow document supplied ended at item 37 ("Freeze the selected model design and hyperparameters"). If later items exist (interpretation, deployment, monitoring), send them and they will be added.*